# Sesión 6 — MUBD

**Clasificación supervisada (I): de los vecinos a las probabilidades**

**Pregunta de partida:** ¿podemos asignar una clase a un cliente o producto a partir de sus características conocidas?

En S3 y S4 trabajamos con respuestas numéricas y probabilidades; en S5 exploramos grupos sin etiquetas. Ahora conocemos la clase de los ejemplos de entrenamiento y estudiaremos cómo clasificar nuevos registros mediante **KNN** y **Naive Bayes**.

**Recorrido:** clientes y KNN → distancias, escalas y votos → productos y Naive Bayes → probabilidades posteriores → comparación de variantes.

**Cómo trabajar:** sigue los pasos, ejecuta cada ejemplo y contrasta la lectura con la salida. Los ejercicios **6-1 y 6-2** quedan sin resolver; sus funciones pendientes no impiden continuar.

**Regla del análisis:** las filas reservadas se consultan en varias comparaciones. Si se usan para elegir una configuración, son **validación**, no un test final independiente. No presentaremos una cifra observada como rendimiento garantizado; también señalaremos las transformaciones y filtros que usan toda la muestra.

Selecciona el kernel del proyecto y ejecuta en orden con **Shift + Enter**. Usa **Variables** y **Esquema / Outline** para orientarte. Para repetir desde cero, usa **Reiniciar kernel → Ejecutar todo**. [Guía de notebooks en VS Code](https://code.visualstudio.com/docs/datascience/jupyter-notebooks).

## 6.0. Preparación

1. Selecciona el kernel del entorno del proyecto. Las bibliotecas ya deben estar instaladas.
2. Ejecuta las primeras celdas **en orden**: localizan la carpeta del proyecto, importan las herramientas y fijan `SEED = 12345`. La semilla permite repetir las particiones aleatorias.
3. Los archivos `s6_knn.py`, `s6_bayes.py`, `s6_seleccion.py`, `s6_evaluacion.py`, `s6_graficos.py` y `s6_controles.py` contienen funciones auxiliares. No hace falta estudiarlos ahora: en el notebook se indica qué hace cada función cuando aparece.

**Convención para leer los resultados:** las matrices muestran la predicción en las filas y la clase real en las columnas. Además del acierto global, observaremos la sensibilidad de la clase minoritaria. En KNN también puede haber casos «Sin decisión»: entonces distinguiremos el acierto entre los casos clasificados, la cobertura y el acierto sobre todos.

Las filas reservadas (`test_knn` y `test_nb`) se usan varias veces para comparar ejemplos. Por tanto, en esta sesión sirven como **evaluación/validación**, no como test final independiente.

In [ ]:
from pathlib import Path  # Construye rutas compatibles con macOS, Windows y Linux.
from functools import partial  # Fija argumentos para reutilizar funciones con controles.
import sys  # Permite registrar la carpeta de scripts auxiliares.

for carpeta in (Path.cwd(), *Path.cwd().parents):  # Busca desde la carpeta actual hacia la raíz.
    if (carpeta / "datos" / "edw.csv").is_file():  # Comprueba si esta carpeta parece ser el proyecto.
        PROJECT_PATH = carpeta  # Guarda la carpeta raíz encontrada.
        DATA_PATH = carpeta / "datos"  # Forma la ruta a los archivos de datos.
        break  # Detiene la búsqueda porque ya se encontró el proyecto.
else:  # Se ejecuta si el bucle terminó sin encontrar el archivo.
    raise FileNotFoundError("Abre el notebook dentro de la carpeta LaSalle_Estadistica.")  # Explica cómo corregir la ruta.

SESSION_PATH = PROJECT_PATH / "sesiones" / "sesion6"  # Localiza los scripts de esta sesión.
if str(SESSION_PATH) not in sys.path:  # Evita añadir la misma ruta dos veces.
    sys.path.insert(0, str(SESSION_PATH))  # Hace que Python encuentre esos módulos al importarlos.

In [ ]:
# %pip install numpy pandas matplotlib scipy scikit-learn seaborn ipywidgets threadpoolctl ipykernel

In [ ]:
import numpy as np  # Cálculo numérico y arreglos.
import pandas as pd  # Lectura y manipulación de tablas.
import matplotlib.pyplot as plt  # Creación de gráficos.
from IPython.display import display  # Presentación de tablas y controles en el notebook.
from threadpoolctl import threadpool_limits  # Limita hilos durante cálculos numéricos.

# Pide a Jupyter que dibuje las figuras debajo de la celda.
%matplotlib inline
# Muestra todos los resultados de una celda, no solo el último.
%config InteractiveShell.ast_node_interactivity = "all"
plt.rcParams.update({"figure.figsize": (8, 4), "figure.dpi": 100})  # Ajusta tamaño y resolución iniciales.
pd.set_option("display.max_columns", 30)  # Permite ver hasta 30 columnas de una tabla.
SEED = 12345  # Fija una semilla para repetir muestras aleatorias.

In [ ]:
pd.DataFrame({  # Construye una tabla breve con las rutas usadas por el cuaderno.
    "Ruta": [str(PROJECT_PATH), str(DATA_PATH), str(SESSION_PATH)],  # Convierte rutas a texto para mostrarlas.
}, index=["Proyecto", "Datos", "Scripts auxiliares"])  # Nombra cada fila para identificar cada ruta.

In [ ]:
from s6_evaluacion import tabla_confusion, evaluar  # Importa la matriz y métricas de evaluación de clasificación.

## 6.1. KNN: bajas de clientes

**Pregunta:** a partir de las características de un cliente, ¿lo clasificamos como permanencia (`churn = 0`) o baja (`churn = 1`)? KNN conserva los registros de entrenamiento y asigna una clase según sus vecinos más próximos.

1. **Preparar:** inspecciona las variables, retira identificadores y codifica las categorías.
2. **Clasificar:** separa entrenamiento y evaluación; calcula distancias y votos para los registros reservados.
3. **Interpretar:** compara acierto y sensibilidad, y explora qué cambia al modificar vecinos, escalas y pesos.

Para un cliente $x$, la distancia euclídea y el voto uniforme son

$$
d(x,z)=\sqrt{\sum_{j=1}^p(x_j-z_j)^2},\qquad
\widehat p(c\mid x)=\frac{\sum_{i\in N_k(x)}\mathbf{1}(y_i=c)}{|N_k(x)|},\qquad
\widehat y(x)=\arg\max_c\widehat p(c\mid x).
$$

$N_k(x)$ contiene los vecinos de entrenamiento más cercanos; si hay empates en la distancia límite, puede incluir más de $k$. Aquí $k$ es un número de vecinos, no de grupos. El voto resume semejanza local: un valor 1 no significa certeza de que el cliente se dará de baja.

Comenzamos con las unidades originales. Como la distancia suma diferencias al cuadrado, una variable medida en decenas de miles puede dominar a otra medida entre 0 y 1. Más adelante compararemos escalas y pesos sin sustituir el ejemplo inicial.

### Diccionario de variables

| Variable | Contenido |
| --- | --- |
| `age`, `annualincome` | Edad e ingresos anuales |
| `calldroprate`, `callfailurerate` | Tasas de llamadas perdidas y cortadas |
| `callingnum`, `customerid` | Teléfono e identificador del cliente |
| `customersuspended` | Indicador de suspensión del cliente |
| `education`, `gender`, `maritalstatus` | Formación, sexo y estado civil |
| `homeowner` | Propiedad de la vivienda |
| `monthlybilledamount` | Factura mensual |
| `noadditionallines` | Campo sin información útil en este archivo |
| `numberofcomplaints`, `numberofmonthunpaid` | Quejas y meses sin pagar |
| `numdayscontractequipmentplanexpiring` | Días hasta el vencimiento del contrato |
| `occupation` | Ocupación |
| `penaltytoswitch` | Penalización por cambio de línea |
| `state` | Estado de Estados Unidos |
| `totalminsusedinlastmonth` | Minutos utilizados el último mes |
| `unpaidbalance` | Saldo pendiente de pago |
| `usesinternetservice`, `usesvoiceservice` | Servicios de internet y voz contratados |
| `percentagecalloutsidenetwork` | Proporción de llamadas fuera de la red |
| `totalcallduration`, `avgcallduration` | Duración total y media de las llamadas |
| `churn` | Baja del cliente: 0 o 1 |
| `year`, `month` | Año y mes |

### 6.1.1. Leer e inspeccionar los datos

Antes de calcular distancias, hay que saber qué contiene la tabla y qué significa cada fila.

1. Lee `edw.csv`; la coma separa las columnas.
2. Revisa cuántas filas y columnas hay, los nombres y los tipos de datos.
3. Cuenta los valores ausentes y observa los rangos de las variables numéricas.
4. Localiza la respuesta `churn`: `0` significa que el cliente permanece y `1` que se da de baja.

**Importante sobre las filas:** cada fila representa un registro mensual, no necesariamente un cliente distinto. El mismo `customerid` puede aparecer en varios meses; `customerid` y `callingnum` son identificadores, no medidas de semejanza. La partición aleatoria de esta práctica puede dejar meses del mismo cliente tanto en entrenamiento como en evaluación. Por ello, no mide cómo funcionaría el modelo con clientes completamente nuevos ni en meses futuros.

In [ ]:
d0 = pd.read_csv(DATA_PATH / "edw.csv")  # Lee los registros mensuales y conserva la tabla original.
resumen_clientes = pd.Series({  # Agrupa cifras básicas de la muestra en un objeto etiquetado.
    "Registros mensuales": len(d0),  # Cuenta filas, incluidas las repeticiones mensuales.
    "Columnas (incluida churn)": d0.shape[1],  # Cuenta variables y respuesta.
    "Clientes distintos": d0["customerid"].nunique(),  # Cuenta IDs diferentes.
    "Clientes con varios registros": d0["customerid"].value_counts().gt(1).sum(),  # Cuenta IDs repetidos.
}, name="Valor")  # Nombra la columna de resultados.
display(resumen_clientes.to_frame())  # Muestra las cifras como tabla vertical.
print("Primeras filas: identificador, mes, variables ilustrativas y respuesta churn.")  # Presenta la vista siguiente.
columnas_vista = ["customerid", "year", "month", "age", "annualincome", "churn"]  # Elige columnas representativas.
display(d0[columnas_vista].head())  # Muestra cinco registros sin saturar la pantalla.

In [ ]:
# Resume las 93 características numéricas, una variable por fila.
excluir_resumen = ["customerid", "callingnum", "year", "month", "churn"]  # Excluye IDs, calendario y respuesta.
numericas_resumen = d0.drop(columns=excluir_resumen).select_dtypes(include="number")  # Se queda con medidas numéricas.
resumen_numerico = numericas_resumen.describe().T[["mean", "std", "min", "50%", "max"]].round(2)  # Resume distribución.
resumen_numerico.columns = ["Media", "Desviación típica", "Mínimo", "Mediana", "Máximo"]  # Traduce títulos.
print("Variables numéricas candidatas a predictor:")  # Anuncia la tabla numérica.
display(resumen_numerico)  # Muestra una variable por fila.

categoricas_d0 = d0.select_dtypes(exclude="number")  # Separa columnas categóricas.
resumen_categorico = pd.DataFrame({  # Construye resumen de niveles.
    "Niveles distintos": categoricas_d0.nunique(),  # Cuenta categorías por variable.
    "Nivel más frecuente": categoricas_d0.mode().iloc[0],  # Identifica la moda.
    "Frecuencia": categoricas_d0.apply(lambda columna: columna.value_counts().iloc[0]),  # Cuenta su aparición.
})  # Termina la tabla categórica.
print("Variables categóricas:")  # Anuncia la tabla de categorías.
display(resumen_categorico)  # Muestra niveles y frecuencias.

ausentes_d0 = d0.isna().sum()  # Cuenta NaN por columna.
ausentes_d0 = ausentes_d0.loc[ausentes_d0 > 0].rename("Valores ausentes").to_frame()  # Oculta columnas sin NaN.
if ausentes_d0.empty:  # Comprueba si hay ausentes.
    print("No se detectan valores NaN; revisa códigos de ausencia escritos como texto.")  # Aclara la limitación.
else:  # Ejecuta si alguna columna tiene NaN.
    display(ausentes_d0)  # Muestra únicamente sus recuentos.

**Lectura de la tabla:** hay **20 468 registros mensuales**, pero solo **9 525 clientes distintos**; **9 430 clientes** aparecen en más de un registro. No se detectan valores ausentes con `isna()`, aunque `noadditionallines` contiene el texto `\N` en todas las filas; al no variar, no ayuda a distinguir clientes.

Las variables también están en escalas muy diferentes: la desviación típica de `annualincome` es aproximadamente **72 129**, mientras que las tasas de llamadas varían en centésimas. Como KNN calcula distancia usando diferencias numéricas, una variable con valores grandes puede dominar la comparación. Más adelante veremos qué cambia al escalar.

### 6.1.2. Preparar las variables

KNN calcula distancias numéricas, así que primero decidimos qué columnas usar y cómo representar las categorías.

1. Retira identificadores y campos que no aportan información útil para comparar clientes. Se trabaja sobre una copia para conservar intacta la tabla original.
2. Declara los niveles educativos en un orden razonable. El orden permite distinguir niveles, pero al codificarlos como números también se supone que los pasos entre niveles tienen la misma distancia.
3. Convierte `occupation` en dos indicadores 0/1. La categoría omitida (`Others`) queda como referencia: ambos indicadores a 0 significan que el registro pertenece a esa categoría.
4. Conserva `churn` como respuesta, separada de las entradas que se usarán para calcular distancias.

Esta codificación sirve para estudiar KNN, pero no es universal: las categorías nominales no tienen un orden natural y asignarles números puede crear distancias artificiales.

In [ ]:
variables_retiradas = ["customerid", "noadditionallines", "state", "year", "month", "callingnum"]  # Campos que no medirán proximidad.
d = d0.drop(columns=variables_retiradas).copy()  # Crea una copia sin alterar d0.

niveles_educacion = ["High School or below", "Bachelor or equivalent", "Master or equivalent", "PhD or equivalent"]  # Orden educativo elegido.
d["education"] = pd.Categorical(d["education"], categories=niveles_educacion, ordered=True)  # Declara educación ordinal.

d["occupationTech"] = d["occupation"].eq("Technology Related Job").astype(int)  # Vale 1 para ocupación tecnológica.
d["occupationNonTech"] = d["occupation"].eq("Non-technology Related Job").astype(int)  # Vale 1 para otra ocupación técnica.
d = d.drop(columns="occupation")  # Retira el texto original ya reemplazado por dos indicadores.
d = d[[c for c in d.columns if c != "churn"] + ["churn"]]  # Coloca la respuesta al final para identificarla.
d.head()  # Inspecciona la tabla codificada.

KNN requiere entradas numéricas. Asignamos códigos desde 1 y guardamos su correspondencia en `codificacion`.

**Ten en cuenta:** los códigos de categorías nominales son etiquetas, no magnitudes. Asignar 1, 2 y 3 crea una distancia artificial; en educación respetamos el orden declarado, pero también suponemos una separación unitaria entre niveles. `Categorical.codes` reserva −1 para categorías ausentes o desconocidas; al sumar 1 se obtiene 0, que debe revisarse y no interpretarse automáticamente como un nivel válido.

En el ejercicio distinguiremos variables ordinales de nominales mediante `to_dummy`.

In [ ]:
codificacion = {}  # Diccionario donde guardaremos etiqueta → código.
for variable in d.columns:  # Revisa las variables una por una.
    if not (isinstance(d[variable].dtype, pd.CategoricalDtype)  # Detecta categorías ordenadas de pandas.
            or pd.api.types.is_string_dtype(d[variable])):  # O detecta texto normal.
        continue  # Omite variables que ya son numéricas.
    niveles = niveles_educacion if variable == "education" else sorted(d[variable].unique())  # Elige el orden de etiquetas.
    codificacion[variable] = dict(zip(niveles, range(1, len(niveles)+1)))  # Guarda el código de cada etiqueta.
    d[variable] = pd.Categorical(d[variable], categories=niveles).codes + 1  # Convierte etiquetas en enteros desde 1.

print("Diccionario de etiquetas y códigos:")  # Titula la correspondencia que se mostrará.
display(pd.Series(codificacion, name="Código por categoría").to_frame())  # Expone el diccionario en formato tabla.
print("Tipo de dato por variable después de codificar:")  # Titula la comprobación de tipos.
display(d.dtypes.rename("Tipo de dato").to_frame())  # Confirma qué datos recibirá KNN.

### 6.1.3. Separar entrenamiento y evaluación

Para que KNN pueda predecir casos que no ha visto, separamos las filas en dos grupos:

- **Entrenamiento (`train_knn`):** contiene las etiquetas conocidas que KNN consultará para buscar vecinos y votar.
- **Evaluación (`test_knn`):** contiene filas reservadas. Comparamos sus etiquetas reales con las predicciones para medir el resultado.

La máscara asigna cada fila a entrenamiento con probabilidad **0,7**. Es una selección aleatoria: el tamaño final puede no ser exactamente el 70 % y la proporción de clases puede diferir un poco entre grupos. Separamos las características `X` de la respuesta `y`; `churn` nunca debe formar parte de las distancias.

En este notebook volveremos a consultar las etiquetas reservadas al explorar distintas configuraciones. Por eso las trataremos como **evaluación/validación**, no como una prueba final independiente.

In [ ]:
p_train = 0.7  # Proporción esperada de filas de entrenamiento.
rng_knn = np.random.default_rng(SEED)  # Crea un generador reproducible.
train_sel_knn = rng_knn.choice([False, True], size=len(d), p=[1-p_train, p_train])  # Sortea una máscara por fila.
train_knn = d.loc[train_sel_knn].copy()  # Selecciona las filas de entrenamiento.
test_knn = d.loc[~train_sel_knn].copy()  # Reserva las filas restantes para evaluación.

X_train_knn = train_knn.drop(columns="churn")  # Entradas de entrenamiento, sin la respuesta.
X_test_knn = test_knn.drop(columns="churn")  # Entradas reservadas con las mismas columnas.
y_train_knn = train_knn["churn"]  # Etiquetas conocidas para ajustar y votar.
y_test_knn = test_knn["churn"]  # Etiquetas reservadas para evaluar.
# Resume cuántas filas quedaron en cada conjunto.
display(pd.Series({"Entrenamiento": len(train_knn),  # Número de filas usadas para aprender.
                   "Evaluación/validación": len(test_knn)},  # Número de filas reservadas.
                  name="Registros").to_frame())  # Presenta cantidades en una columna.
# Compara frecuencias reales de clase, sin presuponer partición estratificada.
frecuencias_knn = pd.DataFrame({"Entrenamiento": y_train_knn.value_counts().sort_index(),  # Cuenta clases en train.
                                "Evaluación/validación": y_test_knn.value_counts().sort_index()})  # Cuenta clases reservadas.
display(frecuencias_knn.rename_axis("churn real"))  # Etiqueta las filas con la clase real.

escalas_train_knn = X_train_knn.std(ddof=1).sort_values(ascending=False)  # Calcula y ordena desviaciones típicas.
display(escalas_train_knn.head().rename("Desviación típica en train").to_frame())  # Muestra cinco escalas mayores.

**Lectura de la partición:** quedaron **14 196 registros** para entrenamiento y **6 272** para evaluación. Entre las filas reservadas, **5 707** son permanencias y **565** bajas; por tanto, solo el **9,01 %** corresponde a `churn = 1`. Esta diferencia entre clases se llama **desequilibrio**.

Después de retirar las columnas indicadas quedan **23 predictores**. Sus escalas son muy distintas: `annualincome` tiene una desviación típica de **72 262,83**, mientras `totalcallduration` tiene **1 954,84**. En la distancia original, las variables con números grandes pueden pesar mucho más; eso no significa que sean mejores para predecir.

### Herramientas KNN: buscar vecinos y reutilizar el cálculo

La clase `Vecinos` prepara una búsqueda de vecinos y guarda sus índices y distancias. Así no hace falta volver a calcular todas las distancias cada vez que cambiamos la regla de voto.

- **`probabilidades(k, kernel)`:** cuenta cuánto apoya cada clase entre los vecinos. Con `kernel="uniforme"`, cada vecino aporta lo mismo; con otros kernels, los pesos pueden diferir.
- **`predecir(k, kernel, minimo)`:** elige la clase con mayor voto. En un empate uniforme usa la semilla fijada. Si se indica un mínimo de votos, algunos casos pueden quedar sin decisión.
- **Empates de distancia:** si varios registros están a la misma distancia que el vecino número `k`, todos pueden participar; por eso a veces intervienen más de `k` vecinos.

La opción `estandarizar=True` divide las columnas por sus desviaciones calculadas en entrenamiento y aplica esos mismos valores a las consultas. No resta la media: restar el mismo centro a todos los registros no cambia las distancias entre ellos. El nombre del kernel `"óptimo"` identifica una fórmula de este ejemplo; no garantiza que sea la mejor para estos datos.

La implementación está en [s6_knn.py](s6_knn.py); las funciones para mostrar resultados están en [s6_graficos.py](s6_graficos.py).

In [ ]:
import importlib  # Permite recargar un módulo sin reiniciar el kernel.
import s6_graficos  # Importa las funciones gráficas de esta sesión.
_ = importlib.reload(s6_graficos)  # Actualiza funciones de s6_graficos desde el archivo guardado.
from s6_knn import Vecinos as _Vecinos, barrido_vecinos  # Importa búsqueda y resumen de vecinos.
from s6_graficos import dibujar_barrido, dibujar_confusion  # Importa vistas de resultados.

Vecinos = partial(_Vecinos, semilla=SEED)  # Fija la semilla sin cambiar la llamada Vecinos(X, y, consulta).

### 6.1.4. Clasificar con 1-NN

**1-NN** significa «un vecino más próximo»: para cada fila reservada, buscamos en entrenamiento el registro con menor distancia y copiamos su `churn` como predicción. En este primer ejemplo usamos las variables con sus unidades originales y no aplicamos pesos.

La matriz muestra la clase predicha en filas y la clase real en columnas:

- Fila `0`, columna `1`: una baja real que el modelo predijo como permanencia (**baja no detectada**).
- Fila `1`, columna `0`: una permanencia real que el modelo predijo como baja (**falsa alarma**).
- La diagonal contiene las predicciones correctas.

Compara el acierto global con la sensibilidad de `churn = 1`, que indica qué proporción de las bajas reales detecta el modelo. En los datos reservados hay pocas bajas, así que un acierto alto no garantiza que las encuentre.

In [ ]:
vecinos_originales = Vecinos(X_train_knn, y_train_knn, X_test_knn)  # Busca y guarda vecinos en escala original.
knn1 = vecinos_originales.predecir(k=1)  # Predice con un solo vecino.
t_knn1 = tabla_confusion(y_test_knn, knn1)  # Compara predicción con clase real.
print("Matriz de confusión · predicción en filas, realidad en columnas:")  # Explica cómo leer la tabla siguiente.
display(t_knn1)  # Muestra los recuentos por clase con un título claro.
dibujar_confusion(t_knn1, "KNN 1-NN · clientes reservados")  # Añade matriz coloreada.
acierto_knn1 = np.mean(knn1.to_numpy() == y_test_knn.to_numpy())  # Calcula fracción global de aciertos.
sensibilidad_knn1 = t_knn1.loc[1, 1] / t_knn1[1].sum()  # Divide bajas detectadas entre todas las bajas reales.
metricas_knn1 = evaluar(y_test_knn, knn1)  # Calcula acierto entre clasificados, cobertura y acierto sobre todos.
metricas_knn1.loc["Sensibilidad churn=1"] = sensibilidad_knn1  # Añade el porcentaje de bajas reales que el modelo detecta.
metricas_knn1 = metricas_knn1.rename({  # Cambia las etiquetas técnicas por nombres fáciles de interpretar.
    "Acierto entre clasificadas": "Acierto global",  # Nombra la proporción de predicciones correctas.
    "Cobertura": "Cobertura",  # Indica la proporción de registros con una predicción.
    "Acierto sobre todas las observaciones": "Acierto sobre todos los registros",  # Aclara el denominador de esta medida.
})  # Termina el cambio de nombres.
print("Métricas de evaluación · valores en porcentaje:")  # Titula la tabla de resultados.
display((100 * metricas_knn1).rename("Porcentaje (%)").to_frame().round(2))  # Muestra métricas con nombre y dos decimales.

**Lectura de 1-NN:** el modelo acierta **83,53 %** de las **6 272** filas reservadas, pero solo detecta **80 de las 565 bajas reales**: sensibilidad **14,16 %**. En la matriz, las **485** bajas de la celda predicción `0` / realidad `1` son bajas que no detectó; las **548** de predicción `1` / realidad `0` son permanencias señaladas por error.

Esto muestra por qué no basta con decir «acierta el 83,53 %»: la mayoría de los registros son permanencias, así que el acierto global puede ocultar que se pierden muchas bajas y que también hay falsas alarmas. Un solo vecino decide por semejanza bajo estas variables y escalas; no significa que dos clientes sean iguales ni expresa certeza sobre una baja futura.

### 6.1.5. Referencia de la clase mayoritaria

Antes de valorar KNN, probamos una regla deliberadamente simple: **decir siempre la clase más frecuente en entrenamiento**. No mira edad, ingresos ni ninguna otra característica; repite la misma respuesta para todos. No es un modelo que queramos usar, sino una referencia para comprobar si KNN aporta algo.

En las filas reservadas hay **5 707 permanencias (`churn = 0`) de 6 272** (**90,99 %**) y solo **565 bajas (`churn = 1`)** (**9,01 %**). Por eso, decir siempre «permanece» consigue **90,99 % de acierto** sin aprender nada: acierta a la mayoría por ser mayoría. Pero detecta **0 de las 565 bajas**; su sensibilidad para `churn = 1` es **0 %**. Ese es el problema: el acierto puede parecer bueno aunque la regla no encuentre a ningún cliente que se da de baja.

La celda muestra dos cosas:

1. La primera tabla cuenta las clases reales de las filas reservadas. Describe el desequilibrio; **no** se usa para escoger la respuesta de la regla.
2. La segunda elige la clase más frecuente usando solo `y_train_knn` y la aplica a cada fila reservada. Se hace así para no tomar decisiones mirando las respuestas que luego se evalúan.

Compara KNN con esta referencia mirando **tanto el acierto global como la sensibilidad de las bajas**. Superar el 90,99 % de acierto no basta por sí solo: interesa comprobar si detecta bajas y cuántas.

In [ ]:
# Frecuencias de las filas reservadas: describen el desequilibrio de clases.
frecuencias_test = y_test_knn.value_counts().sort_index().rename("Registros").to_frame()  # Cuenta clientes por churn real.
frecuencias_test["Porcentaje (%)"] = 100 * frecuencias_test["Registros"] / len(y_test_knn)  # Convierte cuenta a porcentaje.
frecuencias_test.index.name = "churn real"  # Nombra el índice de las clases.
display(frecuencias_test.round({"Porcentaje (%)": 2}))  # Presenta frecuencias y porcentajes.

clase_mayoritaria = y_train_knn.value_counts().idxmax()  # Decide clase mayoritaria sin mirar test.
acierto_mayoritaria = y_test_knn.eq(clase_mayoritaria).mean()  # Evalúa referencia en filas reservadas.
sensibilidad_mayoritaria = (y_test_knn.eq(1) & (clase_mayoritaria == 1)).sum() / y_test_knn.eq(1).sum()  # Recall de bajas.
display(pd.DataFrame([{  # Resume la regla base.
    "Clase elegida en train": clase_mayoritaria,  # Etiqueta elegida con entrenamiento.
    "Acierto (%)": round(100 * acierto_mayoritaria, 2),  # Proporción de respuestas correctas.
    "Sensibilidad churn=1 (%)": round(100 * sensibilidad_mayoritaria, 2),  # Bajas detectadas.
}]))  # Presenta la referencia en una fila.

### 6.1.6. Cambiar el número de vecinos

Ahora mantenemos los datos y la distancia, pero cambiamos **`k`**, el número de vecinos que votan por la clase de cada cliente. Con `k=1` decide un único vecino; al aumentar `k`, la decisión combina más registros cercanos y suele ser menos sensible a un caso individual.

Probamos valores impares de **1 a 21**. No hay un `k` que sea siempre mejor: valores pequeños pueden ser inestables y valores grandes pueden favorecer a la clase mayoritaria. Por eso el gráfico debe leerse junto con:

- la línea de referencia que predice siempre la clase más frecuente;
- la sensibilidad de `churn = 1`, para comprobar cuántas bajas se detectan;
- el acierto global, que resume todas las filas.

Las distancias ya se calcularon, así que el barrido solo cambia el voto y reutiliza la búsqueda. Si escogiéramos el mejor `k` mirando estas filas reservadas, estaríamos usándolas como validación. Aquí se muestran como exploración y no declaramos un ganador.

In [ ]:
K = np.arange(1, 22, 2)  # Define los once valores impares de vecinos que se compararán.
aciertos_originales = barrido_vecinos(vecinos_originales, y_test_knn, K)  # Cambia k y reutiliza distancias.
dibujar_barrido(aciertos_originales, "KNN con las variables originales",  # Dibuja la curva del acierto.
                referencia=acierto_mayoritaria)  # Añade la regla mayoritaria como referencia.
aciertos_originales  # Muestra la proporción de aciertos por cada k.

**Lectura del barrido:** la referencia que siempre predice permanencia acierta **90,99 %** porque esa es la proporción de permanencias, pero detecta **0 bajas**. El 1-NN con unidades originales acierta **83,53 %**.

Al subir `k`, la clase predicha sale de la votación de más vecinos: el acierto observado llega a **90,40 %** con `k=5` y a **90,99 %** entre `k=11` y `k=21`. Igualar el acierto de la referencia no demuestra que el KNN detecte bajas; hay que mirar también la sensibilidad. Esta curva describe estas filas reservadas y sirve para observar el efecto de `k`, no garantiza el resultado en clientes nuevos.

#### Explorar una predicción y sus vecinos

Este control permite mirar un caso concreto de `test_knn`:

- Cambia **`k`** para ver cuántos vecinos participan en el voto.
- Cambia la **posición del cliente** para elegir otra fila reservada. `iloc` usa posiciones que empiezan en 0: la posición 0 es la primera fila.
- La tabla muestra el `churn` real del caso, el `churn` de sus vecinos y la distancia a cada uno.

La distancia indica cercanía según las variables, códigos y escalas actuales; no indica que dos clientes sean iguales ni que el vecino cause la baja. Si hay empate en la distancia límite, el voto uniforme puede incluir más vecinos que los visibles en la tabla. La sección de resultados muestra el voto agregado que realmente se usó.

Sin el control, el ejemplo fijo es `explorar_knn(k=3, cliente=0)`.

In [ ]:
from s6_graficos import explorar_knn as _explorar_knn  # Importa la vista de un caso y sus vecinos.
from s6_controles import crear_control_knn  # Importa el control para cambiar k y el caso.

explorar_knn = partial(_explorar_knn, ajuste=vecinos_originales,  # Fija búsqueda ya calculada.
    X_train=X_train_knn, y_train=y_train_knn, X_test=X_test_knn, reales=y_test_knn)  # Pasa datos y etiquetas de cada grupo.
control_knn = crear_control_knn(explorar_knn, n_clientes=len(test_knn))  # Ajusta rango de selección al tamaño de test.
display(control_knn)  # Muestra el control interactivo.

#### Micropráctica — sensibilidad de las bajas

Completa el ejemplo para **`k=3`**. La predicción debe usar los tres vecinos más próximos de entrenamiento. Después, usa la matriz para calcular la sensibilidad de `churn = 1`:

$$\text{sensibilidad} = \frac{\text{bajas reales predichas como baja}}{\text{total de bajas reales}}.$$

Muestra también el acierto global y compara ambas medidas: ¿el modelo detecta bien las bajas, aunque acierte muchas filas en total?

Por último, revisa `escalas_train_knn.head()`. Una diferencia de miles en ingresos puede contribuir mucho más a la distancia que una diferencia de centésimas en una tasa. Explica qué implica esto para el KNN original. No elijas un `k` definitivo con estas filas reservadas.

In [ ]:
try:  # Si quedan huecos, captura el error y permite continuar.
    k_bajas_mp = ...  # Sustituye por el k=3 indicado; no modifica el resultado fijo knn1.
    pred_bajas_mp = vecinos_originales.predecir(k=k_bajas_mp)  # Predice con los vecinos guardados.
    tabla_bajas_mp = tabla_confusion(y_test_knn, pred_bajas_mp)  # Compara predicción con churn real.
    sensibilidad_bajas_mp = tabla_bajas_mp.loc[..., ...] / tabla_bajas_mp[...].sum()  # Completa clase positiva y denominador.
    display(tabla_bajas_mp)  # Muestra recuentos por clase.
    display(evaluar(y_test_knn, pred_bajas_mp))  # Muestra métricas con denominadores.
    display(sensibilidad_bajas_mp)  # Muestra proporción de bajas detectadas.
    display(escalas_train_knn.head())  # Revisa las variables de mayor escala en train.
except Exception as error:  # Evita que la práctica pendiente detenga el cuaderno.
    print("Micropráctica pendiente: completa k y la clase positiva en los huecos.")  # Recuerda qué falta.
    print(f"Detalle: {type(error).__name__}: {error}")  # Muestra el error concreto.

### 6.1.8. Exigir votos mínimos y permitir rechazos

Normalmente KNN siempre asigna una clase. Aquí añadimos una opción: **si no hay suficientes votos para apoyar una clase, el modelo puede abstenerse** y devolver «Sin decisión».

Usamos las variables originales, **`k=2`** y **`minimo=2`**. En el caso sencillo, los dos vecinos deben apoyar la clase ganadora. Si varios vecinos empatan en la distancia límite y entran en la votación, el mínimo exigido aumenta para mantener la misma exigencia de acuerdo.

La matriz añade una fila **Sin decisión**. Interpreta las tres métricas así:

- **Acierto entre clasificados:** aciertos divididos solo por los casos a los que se asignó clase.
- **Cobertura:** casos con clase asignada divididos por todos los casos.
- **Acierto sobre todos:** aciertos divididos por todos los casos; los rechazados no cuentan como acierto.

Al rechazar los casos dudosos, el acierto entre clasificados puede subir, pero baja la cobertura. La fila de rechazo no es una clase real. En este ejemplo no escogemos `k` ni el mínimo usando las etiquetas reservadas.

In [ ]:
knn_con_rechazo = vecinos_originales.predecir(k=2, minimo=2)  # Rechaza si no obtiene el mínimo de votos requerido en variables originales.
tabla_rechazo_knn = tabla_confusion(y_test_knn, knn_con_rechazo, incluir_rechazos=True)  # Añade la fila sin decisión.
print("Matriz de confusión · predicción en filas, realidad en columnas:")  # Explica cómo leer la tabla.
display(tabla_rechazo_knn)  # Muestra casos decididos y rechazados.
dibujar_confusion(tabla_rechazo_knn, "KNN original · k=2 y voto mínimo=2")  # Colorea recuentos por clase.
metricas_rechazo_knn = evaluar(y_test_knn, knn_con_rechazo)  # Calcula acierto, cobertura y acierto total.
print("Métricas de la regla con rechazos · valores en porcentaje:")  # Titula resultados.
display((100 * metricas_rechazo_knn).rename("Porcentaje (%)").to_frame().round(2))  # Muestra métricas con nombres claros.

**Lectura de la regla con rechazos:** la regla asigna una clase a **5 148 de 6 272 registros**, así que la cobertura es **82,08 %**; los **1 124 restantes** quedan sin decisión. Entre los casos clasificados acierta **90,60 %**, pero si contamos también los rechazados como casos no acertados, el acierto sobre todos es **74,36 %**.

La cobertura alta no significa que detecte las bajas: solo identifica **4 de las 565 bajas reales** y deja **128 bajas** sin decisión. Por eso hay que mirar juntos acierto, cobertura y sensibilidad; un porcentaje alto calculado solo sobre los casos aceptados puede ocultar cuántos casos importantes se rechazaron.

### 6.1.9. Ponderar vecinos y ajustar escalas

Hasta ahora, todos los vecinos incluidos en la votación aportaban lo mismo. Con un **voto ponderado**, cada vecino aporta un peso $w_i$: un peso mayor hace que su clase influya más en la decisión. La proporción de voto para una clase es

$$
\widehat p(c\mid x)=\frac{\sum_{i\in N_k(x)}w_i\,\mathbf{1}(y_i=c)}{\sum_{i\in N_k(x)}w_i}.
$$

El denominador suma los pesos para que los votos de todas las clases sumen 1. Por ejemplo, si los vecinos cercanos de un cliente tienen más peso, sus clases cuentan más que las de vecinos menos cercanos. El modelo predice la clase con mayor voto. **Ese voto normalizado sirve para decidir, pero no es automáticamente una probabilidad fiable o calibrada.**

En este cuaderno hay tres reglas de voto ponderado:

- **`"óptimo"`:** asigna pesos según el orden de cercanía: usa el rango del vecino, no cuánto más lejos está en unidades de distancia. «Óptimo» es el nombre de la fórmula implementada; no garantiza que sea la mejor regla para estos datos.
- **`"triangular"`:** calcula la distancia relativa $u$ de cada vecino respecto al siguiente vecino que queda fuera de los primeros $k$. Cuanto menor es $u$, mayor es el peso, calculado como $1-u$.
- **`"epanechnikov"`:** usa ese mismo $u$, pero convierte la distancia en peso con $0{,}75(1-u^2)$. También favorece a los más cercanos, con una curva distinta.

Antes de comparar estas reglas, ajustamos la escala. La distancia euclídea puede quedar dominada por una variable grande, como los ingresos. Aquí dividimos cada predictor por su desviación típica **calculada con entrenamiento** y aplicamos esa misma escala a los registros reservados. Así hacemos más comparables las variaciones de las columnas; no restamos la media, porque restar el mismo centro a ambos conjuntos no cambia las distancias entre ellos.

1. Aprende las desviaciones típicas en entrenamiento y escala entrenamiento y evaluación con esos mismos valores.
2. Busca y guarda los vecinos en esa nueva escala.
3. Compara los resultados. Para separar el efecto de la escala del efecto de los pesos, observa que con **`k=1` solo vota un vecino**: los distintos kernels no pueden cambiar esa votación; la diferencia frente al 1-NN original se debe al cambio de escala.

Los códigos numéricos de categorías nominales siguen teniendo distancias artificiales, incluso después de escalar. La comparación es exploratoria: se reutilizan las mismas filas reservadas para varias configuraciones.

In [ ]:
# Aprende desviaciones con entrenamiento; conserva el ejemplo 1-NN con kernel óptimo.
vecinos_kernel = Vecinos(X_train_knn, y_train_knn, X_test_knn, estandarizar=True)  # Estandariza con train y calcula vecinos.
kknn1 = vecinos_kernel.predecir(k=1, kernel="óptimo")  # Predice con la ponderación indicada.
t_kknn1 = tabla_confusion(y_test_knn, kknn1)  # Compara clases predichas y reales.
t_kknn1  # Muestra matriz de confusión.
dibujar_confusion(t_kknn1, "KNN estandarizado · k=1")  # Añade vista coloreada.
recall_kernel1 = t_kknn1.loc[1, 1] / t_kknn1[1].sum()  # Calcula sensibilidad de bajas.
display(pd.DataFrame([{  # Presenta medidas en una fila.
    "Acierto global (%)": round(100 * kknn1.eq(y_test_knn).mean(), 2),  # Fracción de aciertos.
    "Sensibilidad churn=1 (%)": round(100 * recall_kernel1, 2),  # Bajas detectadas.
    "Cobertura (%)": 100.0,  # El modelo decide todos los casos.
}]))  # Muestra el resumen.

In [ ]:
aciertos_kernel = barrido_vecinos(vecinos_kernel, y_test_knn, K, kernel="óptimo")  # Reutiliza búsqueda para cada k.
dibujar_barrido(aciertos_kernel, "KNN ponderado · variables originales",  # Dibuja acierto por k.
                referencia=acierto_mayoritaria)  # Contrasta con regla mayoritaria.
aciertos_kernel  # Muestra tabla de acierto para los once valores.

**Lectura de KNN estandarizado:** al dividir cada característica por su desviación típica de entrenamiento, reducimos el dominio de los ingresos sobre la distancia. Con `k=1`, el modelo acierta **94,08 %** y detecta **384 de las 565 bajas** (**67,96 %** de sensibilidad), frente al 1-NN con unidades originales.

Con un solo vecino no hay varios votos que ponderar: los distintos kernels no pueden cambiar qué clase aporta el único voto. Por eso esta diferencia respecto al KNN original se debe al **cambio de escala**, no al kernel. Al aumentar `k`, sí pueden cambiar los pesos relativos de los vecinos; el barrido muestra esa combinación. Igualar escalas tampoco convierte los códigos de categorías nominales en distancias naturales.

#### Micropráctica — rechazos y votos ponderados

En las variables originales, aplica la regla de rechazo con **`k=3` y mínimo de 2 votos**. Muestra la matriz con **Sin decisión** y las tres métricas de `evaluar`; compara el acierto entre clasificados con la cobertura.

Después obtén los votos del kernel triangular con `k=3` y comprueba que cada fila suma 1. Explica por qué esa normalización no demuestra calibración ni certeza.

In [ ]:
try:  # Captura errores mientras el alumno completa los valores pendientes.
    k_rechazo_mp = ...  # Sustituye por k=3, como pide la práctica.
    minimo_rechazo_mp = ...  # Sustituye por el mínimo de 2 votos.
    pred_rechazo_mp = vecinos_originales.predecir(k=k_rechazo_mp, minimo=minimo_rechazo_mp)  # Predice o rechaza cada fila en escala original.
    tabla_rechazo_mp = tabla_confusion(y_test_knn, pred_rechazo_mp, incluir_rechazos=True)  # Añade fila de rechazos.
    display(tabla_rechazo_mp)  # Muestra predicción frente a clase real.
    display(evaluar(y_test_knn, pred_rechazo_mp))  # Separa acierto, cobertura y acierto total.
    votos_kernel_mp = vecinos_kernel.probabilidades(k=..., kernel="triangular")  # Completa k=3 para el kernel triangular.
    display(votos_kernel_mp.head())  # Enseña votos de las primeras filas reservadas.
    display(votos_kernel_mp.sum(axis=1).head().rename("Suma de votos"))  # Comprueba normalización por fila.
except Exception as error:  # Evita que la práctica pendiente detenga el cuaderno.
    print("Micropráctica pendiente: completa vecinos y votos mínimos; puedes continuar.")  # Indica que se puede seguir.
    print(f"Detalle: {type(error).__name__}: {error}")  # Muestra causa concreta si el alumno ya modificó el código.

### 6.1.10. Cambiar la escala y la distribución de las entradas

KNN decide quién es «cercano» calculando distancias entre filas. Por eso, **la forma de expresar las variables puede cambiar qué clientes son vecinos**, aunque los datos y las etiquetas sean los mismos. Aquí comparamos la escala original con dos transformaciones, manteniendo las mismas filas y `k=1`.

1. **Escala original:** los ingresos pueden variar en decenas de miles, mientras que algunas tasas varían entre 0 y 1. Al elevar las diferencias al cuadrado y sumarlas, una variable con números grandes puede dominar la distancia. Esto no significa que sea más importante para predecir la baja.
2. **Estandarización:** para cada variable, se resta su media de entrenamiento y se divide por su desviación típica de entrenamiento:

   $$z_j=\frac{x_j-\bar x_{j,\mathrm{train}}}{s_{j,\mathrm{train}}}.$$

   Así, las variables quedan expresadas en una escala comparable: una diferencia de una unidad equivale aproximadamente a una desviación típica de esa variable. Aprendemos media y desviación solo con entrenamiento y usamos esos mismos valores para transformar las filas reservadas. Centrar ambos conjuntos con la misma media no cambia las distancias entre ellos; dividir por la desviación sí puede cambiar qué vecinos quedan más cerca.
3. **Transformación por cuantiles:** primero se estima, con entrenamiento, qué proporción de observaciones queda por debajo de cada valor. Después, esa proporción se convierte a una escala normal mediante $\Phi^{-1}(\widehat F_{j,\mathrm{train}}(x_j))$. En palabras: sustituye el valor original por su posición relativa en la distribución y la expresa en una escala nueva. Conserva el orden de los valores, pero comprime zonas muy densas y separa más las zonas poco densas; por eso también puede cambiar los vecinos.

En ambos casos, **la transformación se ajusta solo con entrenamiento** y luego se aplica sin reajustar a las filas reservadas. Esto evita que información de evaluación intervenga en el aprendizaje de la transformación. Compara matrices, acierto y sensibilidad de `churn = 1`; como se prueban varias representaciones sobre las mismas filas reservadas, los resultados son exploratorios y no una evaluación final independiente.

Estas transformaciones no prueban que los datos sean normales ni resuelven el problema de asignar números a categorías nominales: sus distancias siguen siendo artificiales.

In [ ]:
# Aprende media y desviación solo con train; churn queda fuera de las entradas.
centros_train_knn = X_train_knn.mean()  # Calcula media por predictor en train.
desviaciones_train_knn = X_train_knn.std(ddof=1).replace(0, 1)  # Calcula desviación muestral y evita dividir por cero.
X_train_z_knn = (X_train_knn - centros_train_knn) / desviaciones_train_knn  # Estandariza train con sus parámetros.
X_test_z_knn = (X_test_knn - centros_train_knn) / desviaciones_train_knn  # Aplica esos mismos parámetros a test.
resumen_z_knn = pd.DataFrame({  # Compara el resultado de estandarizar train.
    "Media (train)": X_train_z_knn.mean(),  # Media de cada predictor tras escalar.
    "Desviación típica (train)": X_train_z_knn.std(ddof=1),  # Desviación de cada predictor tras escalar.
})  # Termina la tabla resumen.
resumen_z_knn["Media (train)"] = resumen_z_knn["Media (train)"].mask(  # Corrige residuos de redondeo.
    resumen_z_knn["Media (train)"].abs() < 0.0005, 0  # Interpreta medias casi cero como cero.
)  # Termina el ajuste de residuos numéricos.
display(resumen_z_knn.head().round(3))  # Muestra cinco variables y tres decimales.

# Centrar ambos conjuntos igual no altera las distancias de la caché ya estandarizada.
pred_z_knn = vecinos_kernel.predecir(k=1, kernel="óptimo")  # Reutiliza el KNN estandarizado.
t_z_knn = tabla_confusion(y_test_knn, pred_z_knn)  # Calcula matriz de confusión.
t_z_knn  # Muestra la matriz por clase.
dibujar_confusion(t_z_knn, "KNN estandarizado · k=1")  # Añade su versión gráfica.
metricas_z_knn = evaluar(y_test_knn, pred_z_knn)  # Calcula acierto y cobertura.
metricas_z_knn.loc["Sensibilidad churn=1"] = t_z_knn.loc[1, 1] / t_z_knn[1].sum()  # Añade recall de bajas.
display((100 * metricas_z_knn).rename("Porcentaje (%)").to_frame().round(2))  # Presenta métricas en porcentaje.

In [ ]:
from sklearn.preprocessing import QuantileTransformer  # Importa el transformador por cuantiles.
# Aprende cuantiles en entrenamiento; conserva índices, columnas y partición.
cuantiles_knn = QuantileTransformer(n_quantiles=min(1000, len(X_train_knn)),  # Limita rejilla al tamaño de train.
                                   output_distribution="normal",  # Mapea cuantiles a una forma normal estándar.
                                   subsample=None, random_state=SEED)  # Usa todas las filas y fija reproducibilidad.
with threadpool_limits(limits=1):  # Limita hilos durante el ajuste.
    X_train_q_knn = pd.DataFrame(cuantiles_knn.fit_transform(X_train_knn),  # Aprende y transforma train.
                               index=X_train_knn.index, columns=X_train_knn.columns)  # Conserva etiquetas e índices.
    X_test_q_knn = pd.DataFrame(cuantiles_knn.transform(X_test_knn),  # Transforma test sin volver a ajustar.
                              index=X_test_knn.index, columns=X_test_knn.columns)  # Mantiene alineación con y_test.
# La nueva geometría requiere una búsqueda; no recalcula los KNN anteriores.
vecinos_cuantiles_knn = Vecinos(X_train_q_knn, y_train_knn, X_test_q_knn, max_k=1)  # Busca vecino en escala nueva.
pred_q_knn = vecinos_cuantiles_knn.predecir(k=1, kernel="óptimo")  # Predice cada fila reservada.
t_q_knn = tabla_confusion(y_test_knn, pred_q_knn)  # Cuenta confusiones por clase.
t_q_knn  # Muestra predicción frente a realidad.
dibujar_confusion(t_q_knn, "KNN con transformación por cuantiles · k=1")  # Presenta matriz coloreada.
metricas_q_knn = evaluar(y_test_knn, pred_q_knn)  # Calcula acierto y cobertura.
metricas_q_knn.loc["Sensibilidad churn=1"] = t_q_knn.loc[1, 1] / t_q_knn[1].sum()  # Añade recall de bajas.
display((100 * metricas_q_knn).rename("Porcentaje (%)").to_frame().round(2))  # Muestra métricas en porcentaje.

# Compara cuartiles de la escala original con los cuantiles transformados.
cuartiles_ingresos = pd.DataFrame({  # Construye tabla con los mismos cuantiles en dos escalas.
    "Original": X_train_knn["annualincome"].quantile([0, .25, .5, .75, 1]),  # Importes originales.
    "Transformado": X_train_q_knn["annualincome"].quantile([0, .25, .5, .75, 1]),  # Valores tras transformar.
}).rename_axis("Cuantil de annualincome").round(2)  # Etiqueta filas y redondea cifras.
cuartiles_ingresos = cuartiles_ingresos.mask(cuartiles_ingresos.abs() < 0.005, 0)  # Evita mostrar -0.00.
display(cuartiles_ingresos)  # Compara ambas escalas.

**Lectura de las transformaciones:** al estandarizar, las primeras entradas tienen media cercana a **0** y desviación **1** en entrenamiento. El resultado reproduce **94,08 %** de acierto y **384/565 bajas detectadas (67,96 %)**: centrar no añade información.

Con cuantiles aprendidos en entrenamiento, 1-NN obtiene **95,26 %** de acierto y detecta **409/565 bajas (72,39 %)**, con **156 falsos negativos**, **141 falsos positivos** y cobertura completa. `annualincome` pasa de un rango **5–249 987** a aproximadamente **−5,20–5,20**; la mediana transformada queda cerca de cero. La diferencia refleja otra geometría, no una solución óptima: se reutilizan las mismas filas reservadas y puede haber clientes compartidos entre conjuntos.

#### Micropráctica — transformar con parámetros de entrenamiento

Reutiliza la media y la desviación aprendidas para transformar `annualincome` en las filas reservadas. Completa la celda y muestra sus cuartiles. ¿Por qué la media transformada no tiene que ser exactamente cero? Compara esta operación lineal con la transformación por cuantiles, sin reajustar el transformador ni buscar otro `k`.

In [ ]:
try:  # Captura huecos aún no completados sin bloquear el cuaderno.
    variable_escala_mp = ...  # Escribe annualincome, la variable pedida.
    ingresos_z_mp = (X_test_knn[variable_escala_mp] - centros_train_knn[variable_escala_mp]) / desviaciones_train_knn[variable_escala_mp]  # Aplica parámetros de train.
    display(ingresos_z_mp.quantile([0, .25, .5, .75, 1]))  # Muestra mínimo, cuartiles y máximo transformados.
    display(ingresos_z_mp.mean())  # Compara la media reservada con el centro aprendido en train.
except Exception as error:  # Entra aquí si quedan huecos o nombres incompletos.
    print("Micropráctica pendiente: completa la variable y reutiliza sus parámetros de entrenamiento.")  # Indica la tarea.
    print(f"Detalle: {type(error).__name__}: {error}")  # Explica el fallo concreto.

## Ejercicio 6-1 — KNN con datos bancarios (sin resolver)

**Objetivo:** construir una clasificación KNN con `bank-additional-full.csv`. La respuesta que queremos predecir es `y` (`yes`/`no`); no debe formar parte de las características. El archivo ya se carga en `banco`, pero preparar las variables, crear la partición y entrenar/evaluar el clasificador son tareas tuyas.

### Pasos de trabajo

1. **Inspecciona:** averigua qué representa cada fila, mira tipos, categorías, valores ausentes y frecuencia de `y`.
2. **Prepara:** separa `y` de las entradas; codifica las categorías nominales como indicadores y las ordinales con un orden justificado. Retira `pdays` como pide el ejercicio y revisa que no aparezcan ausentes al codificar.
3. **Divide:** toma una submuestra reproducible de 10 000 filas y separa entrenamiento y evaluación. Registra el tamaño y las frecuencias de `yes`/`no` en cada parte.
4. **Compara:** ajusta KNN, usa la regla mayoritaria como referencia y estudia varios valores de `k`. Interpreta la matriz y la sensibilidad de `yes`, no solo el acierto global.

**Pistas para evitar errores:** consulta **6.1.1–6.1.6** para inspección, preparación, partición y evaluación; `to_dummy` ayuda a crear indicadores, pero tú decides qué variables son ordinales. Comprueba las categorías antes de codificar: una categoría no incluida en el orden puede convertirse en ausente.

Piensa también cuándo se conoce cada predictor: `duration` solo se conoce después de la llamada, así que no estaría disponible para una predicción hecha antes de llamar. Si utilizas las etiquetas reservadas para escoger `k`, trátalas como validación; no las presentes como una prueba final independiente. Las celdas siguientes contienen pistas, pero no la solución.

In [ ]:
from s6_knn import to_dummy  # Importa ayuda para codificar categorías sin decidir su tipo.
banco = pd.read_csv(DATA_PATH / "bank-additional-full.csv", sep=";")  # Lee el archivo con separador punto y coma.
print(f"Datos preparados: {len(banco):,} filas y {banco.shape[1]} columnas.")  # Informa dimensiones de la tabla.

### 1. Inspección inicial

Antes de modificar el archivo, familiarízate con él. Comprueba el tamaño de `banco`, mira algunas filas y los tipos de las columnas. Cuenta cuántas veces aparece cada valor de `y` y revisa las categorías de las entradas.

Pregúntate: ¿qué representa una fila?, ¿hay categorías como `unknown`?, ¿hay valores ausentes?, ¿qué clase es menos frecuente? Conserva `banco` sin modificar; usarás esa tabla original en los siguientes pasos.

**Consulta:** `shape`, `head()`, `dtypes` y `value_counts()` en **6.1.1**.

In [ ]:
# Apartado 1 — Inspecciona la tabla bancaria; conserva banco sin modificar.
# Consulta pd.read_csv, shape, head, dtypes y value_counts en 6.1.1.
# Muestra tamaño, primeras filas, tipos y frecuencias de la respuesta y.
# Anota una observación sobre las categorías disponibles.
# Escribe aquí tu solución.

### 2. Preparar las variables

KNN necesita números para calcular distancias. Prepara una tabla de entradas numéricas y una respuesta separada:

1. Decide y justifica qué categorías tienen un orden real (por ejemplo, nivel educativo) y cuáles son solo nombres (por ejemplo, profesión).
2. Codifica las ordinales con un orden explícito; convierte las nominales en columnas indicadoras con `to_dummy`. No asignes números consecutivos a una categoría nominal sin explicar que eso crearía un orden y distancias artificiales.
3. Separa `y` como respuesta. Retira `pdays` como pide el ejercicio y no incluyas `y` entre los predictores.
4. Después de codificar, comprueba los tipos y los valores ausentes: una categoría no contemplada puede convertirse en `NaN`.

**Consulta:** preparación de variables y uso de `to_dummy` en **6.1.2**.

In [ ]:
# Apartado 2 — Codifica las variables y retira pdays.
# Consulta los códigos ordinales y las dummies de 6.1.2; usa to_dummy.
# a. Declara un orden defendible para las ordinales; no ordenes unknown sin justificarlo.
# b. Codifica las nominales y reúne las columnas conservando índices únicos.
# c. Separa y como respuesta; revisa tipos, ausentes y escalas de las entradas.
# d. Elimina pdays y no incluyas y en los predictores.
# Escribe aquí tu solución.

### 3. Tomar una submuestra y separar los datos

El archivo bancario es grande; para practicar, toma una **submuestra aleatoria de 10 000 filas sin reemplazo** y fija `SEED = 12345` para que puedas reproducirla.

Después divide esa submuestra en entrenamiento y evaluación. Indica qué proporción usaste y muestra cuántas filas y cuántos `yes`/`no` quedaron en cada grupo. Al ser una selección aleatoria, los tamaños y proporciones observados pueden diferir ligeramente de los esperados.

**Consulta:** separación de `X` e `y` y máscaras en **6.1.3**. Conserva los índices de las filas para que las características y la respuesta sigan alineadas.

In [ ]:
# Apartado 3 — Submuestra reproducible y partición.
# Selecciona 10 000 filas sin reemplazo usando SEED; conserva banco preparado.
# Crea una máscara de entrenamiento y separa predictores y respuesta.
# Muestra tamaños y frecuencias de y; no modifiques train_knn ni test_knn.
# Escribe aquí tu solución.

### 4. Estudiar el número de vecinos

Ajusta KNN usando **solo las filas de entrenamiento** y predice las filas de evaluación. Reutiliza una búsqueda de vecinos para probar varios valores de `k` sin construir una tabla enorme con todas las distancias.

Compara los resultados con la regla mayoritaria: ¿supera KNN la estrategia de repetir siempre la clase más común? Mira la matriz de confusión y la sensibilidad de `yes` para saber cuántas respuestas positivas reales detecta. Un acierto global alto puede ser engañoso si `yes` es poco frecuente.

Al interpretar, considera también la escala de las variables y si cada predictor estaría disponible en el momento real de la predicción. Si miras los resultados de evaluación para escoger `k`, esas filas pasan a ser **validación**; no son un test final independiente.

**Consulta:** `Vecinos`, `barrido_vecinos` y las métricas de clasificación en **6.1.4–6.1.6**.

In [ ]:
# Apartado 4 — Ajustar KNN y comparar el número de vecinos.
# Consulta Vecinos, barrido_vecinos y la referencia mayoritaria en 6.1.5–6.1.6.
# Reutiliza una búsqueda para varios valores de k; no materialices todas las distancias.
# Compara el acierto, la matriz predicción × realidad y la sensibilidad de yes.
# Interpreta escalas, errores y disponibilidad de duration; identifica si test actúa como validación.
# Escribe aquí tu solución.

## 6.2. Naive Bayes: clasificar productos

**Pregunta:** a partir de las características registradas de un producto, ¿qué categoría de `target` predice el modelo? En KNN, la decisión dependía de qué clases tenían los vecinos cercanos. Naive Bayes calcula una puntuación para cada clase combinando:

1. **El prior:** qué frecuente es la clase en los productos de entrenamiento.
2. **La información de las características:** qué tan compatibles son los valores observados con cada clase.

Después transforma esas puntuaciones en posteriores, que suman aproximadamente 1, y predice la clase con posterior más alto. Para hacerlo, Naive Bayes supone que las características son independientes una vez fijada la clase; revisaremos qué significa y por qué puede ser una simplificación fuerte.

| Variable | Contenido y papel |
| --- | --- |
| `id` | Identificador para localizar el producto; no se usa como predictor. |
| `feat_1` … `feat_93` | 93 características numéricas del producto. |
| `target` | Categoría real del producto; es la respuesta, nunca una entrada. |

### 6.2.1. Leer e inspeccionar antes de ajustar

El archivo está separado por `;` y utiliza coma decimal. Antes de modelar, comprueba el tamaño de la tabla, los nombres de las columnas, cuántas clases hay y si `id` identifica una fila única. Revisa también los rangos y valores ausentes.

En este archivo, **cero es un valor observado**, no un dato ausente. Que las columnas sean numéricas tampoco garantiza que sigan una distribución normal ni que sean independientes: eso son supuestos que el modelo hará y que conviene tener presentes al interpretar los resultados.

In [ ]:
# Conserva la tabla original; id localiza el producto y target es la respuesta.
datos = pd.read_csv(DATA_PATH / "products.csv", sep=";", decimal=",")  # Lee el archivo con separador y coma decimal.
resumen_productos = pd.Series({  # Agrupa datos básicos de la tabla.
    "Productos": len(datos),  # Cuenta registros.
    "Columnas totales": datos.shape[1],  # Cuenta todas las columnas.
    "Características predictoras": datos.shape[1] - 2,  # Excluye id y target.
    "Clases target": datos["target"].nunique(),  # Cuenta categorías de respuesta.
    "Identificadores únicos": datos["id"].nunique(),  # Verifica unicidad de id.
}, name="Valor")  # Da un nombre a la columna de resultados.
display(resumen_productos.to_frame())  # Muestra resumen en formato tabla.
print("Vista de ejemplo: identificador, dos características y clase real.")  # Explica la muestra siguiente.
display(datos[["id", "feat_1", "feat_2", "target"]].head())  # Muestra cinco filas representativas.

In [ ]:
# Resume las 93 características numéricas, una variable por fila.
resumen_features = datos.drop(columns=["id", "target"]).describe().T[  # Retira identificador y respuesta.
    ["mean", "std", "min", "50%", "max"]  # Conserva estadísticas útiles para comparar distribuciones.
].rename(columns={"mean": "Media", "std": "Desviación típica",  # Traduce media y desviación.
                  "min": "Mínimo", "50%": "Mediana", "max": "Máximo"})  # Traduce extremos y mediana.
print("Resumen de las 93 características (cero es un valor registrado):")  # Aclara cómo leer los ceros.
display(resumen_features.round(2))  # Redondea valores para facilitar lectura.
# Cuenta observaciones por clase y calcula su porcentaje del conjunto.
frecuencias_target = datos["target"].value_counts().sort_index().rename("Productos").to_frame()  # Cuenta productos por clase.
frecuencias_target["Porcentaje (%)"] = 100 * frecuencias_target["Productos"] / len(datos)  # Calcula porcentaje del total.
frecuencias_target.index.name = "Clase target"  # Identifica etiquetas de fila.
display(frecuencias_target.round({"Porcentaje (%)": 2}))  # Muestra recuento y proporción.

In [ ]:
missings_por_columna = datos.isna().sum()  # Cuenta NaN en cada columna.
resumen_ausentes = missings_por_columna.value_counts().sort_index().rename_axis(  # Cuenta columnas por ausentes.
    "Ausentes por columna"  # Usa un nombre que describe las categorías de fila.
).to_frame("Número de columnas")  # Nombra el recuento de columnas.
display(resumen_ausentes)  # Muestra cuántas columnas tienen 0, 1, ... NaN.

# Cero es un valor observado en este archivo, no un dato ausente.
valores_productos = datos.drop(columns=["id", "target"])  # Aísla las 93 entradas.
resumen_valores = pd.Series({  # Resume el dominio de las características.
    "Celdas con valor cero": int(valores_productos.eq(0).sum().sum()),  # Suma ceros de toda la tabla.
    "Todas las entradas no negativas": bool(valores_productos.ge(0).all().all()),  # Verifica que sean ≥ 0.
    "Todas las entradas enteras": bool(np.equal(valores_productos, np.floor(valores_productos)).all().all()),  # Comprueba enteros.
}, name="Resultado")  # Etiqueta la columna de resultados.
display(resumen_valores.to_frame())  # Muestra cada comprobación en una fila.

**Lectura de los productos:** el archivo contiene **43 190 filas** y **95 columnas**: un identificador, 93 características y la clase `target`, con **9 clases** posibles. Los identificadores son únicos y no se detectan valores ausentes.

Las características son recuentos enteros no negativos y contienen **3 185 067 ceros**. Esos ceros son valores registrados, no ausencias. En el ejercicio los modelaremos con gaussianas continuas para practicar Naive Bayes, aunque esta descripción no demuestra que una gaussiana sea la distribución adecuada.

La tabla de frecuencias muestra además que las clases no aparecen con la misma frecuencia. Si una clase es mucho más común, un modelo puede acertar a menudo prediciéndola y aun así recuperar mal las clases menos frecuentes. Por eso luego miraremos la sensibilidad por clase, no solo el acierto global.

### 6.2.2. Independencia condicional y partición

Naive Bayes supone que, dada la clase, las entradas son independientes:

$$p(x_1,\ldots,x_p\mid C=c)=\prod_j p(x_j\mid C=c).$$

1. Calcula las correlaciones entre características, sin incluir `id` ni `target`.
2. Cuenta cada pareja una sola vez, sin la diagonal.
3. Lee la tabla de salida: muestra solo intervalos ocupados, con número y porcentaje de parejas; el resumen indica rango y umbrales.

La correlación marginal baja no demuestra independencia condicional, y la matriz se calcula sobre toda la muestra, como en el recorrido original. El filtro posterior de correlación reutiliza esta matriz y por ello también tiene carácter exploratorio.

In [ ]:
# Correlaciones entre las 93 entradas; id y target no forman parte del cálculo.
predictores_nb = datos.columns.difference(["id", "target"], sort=False)  # Selecciona solo características.
with threadpool_limits(limits=1):  # Limita hilos durante el cálculo de correlaciones.
    cor_productos = datos[predictores_nb].corr()  # Calcula correlación lineal por parejas.
# Extrae el triángulo superior para contar cada pareja una vez, sin diagonal.
cor_pares = cor_productos.to_numpy()[np.triu_indices(len(predictores_nb), k=1)]  # Obtiene pares no duplicados.

# Agrupa correlaciones en intervalos de ancho 0,1.
cortes_cor = np.linspace(-1, 1, 21)  # Crea 21 límites para formar 20 intervalos.
conteos_cor, _ = np.histogram(cor_pares, bins=cortes_cor)  # Cuenta parejas por intervalo.
etiquetas_cor = [  # Prepara etiquetas legibles para los intervalos.
    f"[{cortes_cor[i]:.1f}, {cortes_cor[i+1]:.1f}{']' if i == len(conteos_cor)-1 else ')'}"  # Etiqueta rango actual.
    for i in range(len(conteos_cor))  # Repite para todos los intervalos.
]  # Termina la lista de etiquetas.
porcentajes_cor = 100 * conteos_cor / len(cor_pares)  # Convierte los recuentos en porcentajes.
tabla_cor = pd.DataFrame({  # Reúne recuentos y porcentajes.
    "Número de parejas": conteos_cor,  # Recuento absoluto por intervalo.
    "Porcentaje del total": [f"{valor:.2f}%".replace(".", ",") for valor in porcentajes_cor],  # Porcentaje en formato español.
}, index=etiquetas_cor).rename_axis("Intervalo de correlación r")  # Usa intervalos como filas.
tabla_cor = tabla_cor.loc[tabla_cor["Número de parejas"] > 0]  # Oculta rangos sin parejas.
display(tabla_cor)  # Muestra solo intervalos observados.

# Resume rango y umbrales utilizados para interpretar la matriz.
parejas_cerca_cero = int((np.abs(cor_pares) <= 0.1).sum())  # Cuenta pares con |r| ≤ 0,1.
parejas_altas = int((cor_pares > 0.8).sum())  # Cuenta pares con r > 0,8.
rango_cor = f"{cor_pares.min():.3f} a {cor_pares.max():.3f}".replace(".", ",")  # Formatea mínimos y máximos.
porcentaje_cerca_cero = f"{100 * parejas_cerca_cero / len(cor_pares):.1f}%".replace(".", ",")  # Porcentaje cercano a cero.
porcentaje_cor_alta = f"{100 * parejas_altas / len(cor_pares):.2f}%".replace(".", ",")  # Porcentaje por encima de 0,8.
conteo_cerca_cero = f"{parejas_cerca_cero:,}".replace(",", " ")  # Separa miles con espacio fino.
conteo_cor_altas = f"{parejas_altas:,}".replace(",", " ")  # Formatea el recuento del umbral alto.
resumen_cor = pd.DataFrame({  # Construye tabla con nombre y resultado.
    "Indicador": ["Parejas de características", "Rango de correlación marginal",  # Incluye tamaño y rango.
                  "Parejas con |r| ≤ 0,1", "Parejas con r > 0,8"],  # Incluye umbrales de lectura.
    "Resultado": [f"{len(cor_pares):,}".replace(",", " "), rango_cor,  # Resultado del tamaño y rango.
                  f"{conteo_cerca_cero} ({porcentaje_cerca_cero})",  # Resultado del umbral cercano a cero.
                  f"{conteo_cor_altas} ({porcentaje_cor_alta})"],  # Resultado del umbral positivo alto.
}).set_index("Indicador")  # Usa cada indicador como etiqueta de fila.
display(resumen_cor)  # Muestra resumen en tabla.
print("Lectura: son asociaciones lineales entre pares de entradas; no prueban independencia condicional por clase.")  # Recuerda límite.

In [ ]:
# Cada barra cuenta una pareja de características; la línea discontinua es el umbral del filtro.
fig, ax = plt.subplots(figsize=(10, 4))  # Crea una figura horizontal.
_ = ax.hist(cor_pares, bins=np.linspace(-1, 1, 21), color="steelblue", edgecolor="white")  # Dibuja frecuencias.
_ = ax.axvline(0, color="gray", linewidth=1, linestyle=":")  # Marca correlación nula.
_ = ax.axvline(0.8, color="firebrick", linewidth=1.5, linestyle="--", label="Umbral positivo 0,8")  # Marca el corte.
_ = ax.set(xlim=(-1, 1), xlabel="Correlación marginal", ylabel="Número de parejas",  # Define escala y nombres de ejes.
           title="Distribución de correlaciones entre características")  # Añade título descriptivo.
_ = ax.grid(axis="y", alpha=0.2)  # Añade guías para comparar barras.
ax.set_axisbelow(True)  # Coloca cuadrícula detrás de los datos.
_ = ax.legend(frameon=False)  # Explica la línea discontinua.
fig.tight_layout()  # Ajusta márgenes.
plt.show()  # Muestra solo el gráfico, no objetos auxiliares.

#### Separar entrenamiento y evaluación

Antes de calcular correlaciones y ajustar Naive Bayes, separamos las filas:

- **Entrenamiento:** se usa para aprender priors y distribuciones de las características.
- **Evaluación (`test_nb`):** se reserva para comparar después las clases predichas con las reales.

Quitamos `id` porque solo identifica el producto y no debe ayudar a decidir su clase. Conservamos `target` aparte como respuesta; si entrara en `X`, el modelo estaría recibiendo la solución como dato de entrada.

La partición asigna cada fila a entrenamiento con probabilidad **0,7**. Es aleatoria, no estratificada ni de tamaño fijo, por lo que las cantidades y proporciones de clase pueden variar ligeramente. En este cuaderno consultaremos `test_nb` al comparar variantes; por ello, estas filas son **evaluación/validación**, no un test final independiente.

In [ ]:
# Separa productos por filas y conserva sus índices originales.
d_productos = datos.drop(columns="id")  # Excluye el identificador de las entradas.
rng_nb = np.random.default_rng(SEED)  # Crea generador reproducible para productos.
train_sel_nb = rng_nb.choice([False, True], size=len(d_productos), p=[0.3, 0.7])  # Sortea máscara con 70% esperado en train.
train_nb = d_productos.loc[train_sel_nb].copy()  # Copia observaciones de entrenamiento.
test_nb = d_productos.loc[~train_sel_nb].copy()  # Copia filas reservadas para evaluación/validación.

# Separa predictores y respuesta para evitar que target entre en el modelo.
X_train_nb, y_train_nb = train_nb.drop(columns="target"), train_nb["target"]  # Entradas y etiquetas de train.
X_test_nb, y_test_nb = test_nb.drop(columns="target"), test_nb["target"]  # Entradas y etiquetas reservadas.
# Informa del tamaño de cada conjunto.
display(pd.DataFrame([{"Entrenamiento": len(train_nb),  # Filas usadas para estimar el modelo.
                       "Evaluación/validación": len(test_nb)}]))  # Filas reservadas para medir resultados.
# Compara cuántos productos hay por clase en cada partición.
frecuencias_nb = pd.DataFrame({"Entrenamiento": y_train_nb.value_counts().sort_index(),  # Frecuencia por clase en train.
                              "Evaluación/validación": y_test_nb.value_counts().sort_index()}).rename_axis("Clase real")  # Frecuencia en test.
display(frecuencias_nb)  # Muestra distribución por clase.
# Elige clases de referencia usando exclusivamente entrenamiento.
clase_mayoritaria_nb = y_train_nb.value_counts().idxmax()  # Clase con más ejemplos.
clase_minoritaria_nb = y_train_nb.value_counts().idxmin()  # Clase con menos ejemplos.
acierto_mayoritaria_nb = y_test_nb.eq(clase_mayoritaria_nb).mean()  # Evalúa regla mayoritaria en filas reservadas.
display(pd.DataFrame([{  # Resume la referencia de clasificación.
    "Clase mayoritaria (train)": clase_mayoritaria_nb,  # Etiqueta más frecuente en train.
    "Clase minoritaria (train)": clase_minoritaria_nb,  # Etiqueta menos frecuente en train.
    "Acierto de la regla mayoritaria (%)": round(100 * acierto_mayoritaria_nb, 2),  # Acierto en evaluación.
}]))  # Cierra la tabla de referencia.

### 6.2.2. ¿Qué significa la independencia condicional?

Naive Bayes simplifica el problema suponiendo que, **si ya conocemos la clase**, cada característica aporta información por separado de las demás. Por ejemplo, al evaluar un producto de `Class_2`, calcula por separado cuánto encajan sus valores de `feat_1`, `feat_2`, etc. con esa clase, y luego combina esas aportaciones.

A esta idea se le llama **independencia condicional**: las características no tienen que ser independientes en general; el supuesto dice que lo serían dentro de cada clase. En la fórmula, la aportación conjunta se aproxima multiplicando las aportaciones individuales:

$$p(x_1,\ldots,x_p\mid C=c)\approx\prod_j p(x_j\mid C=c).$$

Es una simplificación fuerte. Si dos características miden algo parecido, Naive Bayes puede contar evidencia redundante más de una vez. Para explorar si las columnas se parecen, calculamos correlaciones lineales entre las 93 características:

1. Excluimos `id` y `target`: ni el identificador ni la respuesta son características cuya redundancia queramos medir.
2. Contamos cada pareja una sola vez. La diagonal compara cada variable consigo misma y no es una pareja distinta.
3. Leemos el histograma y la tabla: $r$ está entre −1 y 1; cerca de 0 indica poca asociación **lineal**, cerca de 1 una asociación positiva y cerca de −1 una negativa. Cada intervalo muestra cuántas parejas caen en ese rango.

**Límite de esta comprobación:** la correlación se calcula sin separar por clase. Por eso, correlaciones marginales pequeñas no demuestran independencia condicional; solo describen asociaciones lineales entre columnas en la muestra completa. El filtro posterior reutiliza esta matriz y también debe leerse como exploratorio.

### 6.2.3. Ajustar Naive Bayes gaussiano y calcular posteriores

Ajustamos el modelo **solo con los productos de entrenamiento**. Para cada clase y cada característica, el modelo resume los valores observados mediante una media y una desviación típica:

- La **media** señala el centro de los valores de esa característica para esa clase.
- La **desviación típica** indica cuánto suelen alejarse los valores de ese centro.
- El **prior** de una clase es su proporción entre los productos de entrenamiento, antes de observar las características del producto nuevo.

Para clasificar un producto, Naive Bayes combina el prior con las densidades de sus características. La expresión representa ese cálculo para una clase $c$ y las entradas $x_1,\ldots,x_p$:

$$
\widehat p(c\mid x)=\frac{\pi_c\prod_j f_{jc}(x_j)}{\sum_h\pi_h\prod_j f_{jh}(x_j)}.
$$

Primero se obtiene una puntuación para cada clase: prior por el producto de las densidades. Después se divide por la suma de las puntuaciones de todas las clases. Así se obtienen posteriores que suman aproximadamente 1; la predicción es la clase con posterior más alto. Una densidad $f_{jc}(x_j)$ **no es por sí sola una probabilidad de clase** y puede superar 1.

En esta implementación se usan logaritmos para evitar que la multiplicación de muchas cantidades pequeñas se redondee a cero; `logsumexp` permite normalizar las puntuaciones de forma estable. Si una desviación es cero o una densidad se redondea a cero, se sustituye por **0,001**. Los valores ausentes se omiten al predecir; no se imputan durante el ajuste.

**Supuestos y límites de este ejemplo:** las características son recuentos enteros con muchos ceros, pero se aproximan mediante distribuciones gaussianas continuas. Además, el modelo trata las entradas como condicionalmente independientes dada la clase. Estas decisiones permiten estudiar el procedimiento, pero no demuestran que la aproximación describa bien estos datos.

La clase `BayesGaussiano` está implementada en [s6_bayes.py](s6_bayes.py). Las siguientes celdas muestran los priors, los parámetros estimados y ejemplos de predicciones.

In [ ]:
from s6_bayes import BayesGaussiano  # Importa el NB gaussiano implementado para esta sesión.

In [ ]:
nb = BayesGaussiano().fit(X_train_nb, y_train_nb)  # Ajusta NB gaussiano con las 93 entradas de train.
tabla_priors = pd.DataFrame({  # Reúne recuento y probabilidad previa de cada clase.
    "Frecuencia en train": nb.recuentos,  # Número de productos por clase.
    "Probabilidad a priori": nb.prior,  # Frecuencia relativa por clase.
})  # Termina la tabla de priors.
display(tabla_priors.round({"Probabilidad a priori": 4}))  # Redondea las probabilidades.
display(pd.DataFrame({  # Comprueba que los priors forman una distribución válida.
    "Comprobación": ["Suma de probabilidades a priori"],  # Describe el cálculo.
    "Valor": [round(float(nb.prior.sum()), 6)],  # Muestra suma cercana a uno.
}))  # Muestra la comprobación.

In [ ]:
# Cada pareja variable/clase contiene media y desviación, no una probabilidad posterior.
tablas_nb = pd.concat(nb.tablas, names=["Variable", "Clase"])  # Une parámetros de las 93 características.
print(f"Parámetros estimados para {tablas_nb.index.get_level_values('Variable').nunique()} características y {len(nb.clases)} clases: {len(tablas_nb)} filas.")  # Resume el tamaño de la tabla.
print("Ejemplo completo de feat_45 (todas las clases):")  # Anuncia la característica elegida.
display(tablas_nb.loc["feat_45"].round(3))  # Muestra media y desviación por clase.
# La tabla completa permanece en tablas_nb; el selector siguiente permite consultar cualquier variable.

#### Explorar los parámetros de una característica

El ajuste guarda, para cada pareja **característica–clase**, una media y una desviación típica estimadas con los productos de entrenamiento. En el gráfico puedes elegir una característica y comparar sus nueve clases:

- El punto o barra central representa la **media** de esa característica para la clase.
- La barra de error muestra **una desviación típica a cada lado** de la media. Indica dispersión de los productos observados dentro de esa clase; no es un intervalo de confianza ni la incertidumbre de la media.
- Si las medias de varias clases están cerca y sus dispersiones son amplias, esa característica por sí sola puede no separar bien esas clases. Si las medias difieren, podría ayudar, pero aún importa el solapamiento y la relación con las demás características.

Las etiquetas `Class_1`–`Class_9` son nombres de categorías: no tienen orden ni distancia numérica natural. Cambiar la característica del selector solo cambia lo que se visualiza; **no vuelve a ajustar el modelo**.

In [ ]:
from s6_graficos import explorar_tabla_nb as _explorar_tabla_nb  # Importa vista de medias y desviaciones por variable.
from s6_controles import crear_control_tabla_nb  # Importa el selector de característica.

explorar_tabla_nb = partial(_explorar_tabla_nb, modelo=nb)  # Fija el NB entrenado que se quiere inspeccionar.
control_tabla_nb = crear_control_tabla_nb(explorar_tabla_nb, variables=predictores_nb)  # Ofrece una opción por predictor.
display(control_tabla_nb)  # Muestra el selector y su gráfico.

### 6.2.4. Evaluar las predicciones

El modelo ya está ajustado. Ahora le damos las características de los productos reservados y obtenemos una clase predicha para cada uno. Solo después comparamos esas predicciones con `target`, que contiene las clases reales de esas filas.

La matriz de confusión tiene **predicción en filas** y **clase real en columnas**:

- Los valores de la diagonal son productos cuya clase se predijo correctamente.
- Los valores fuera de la diagonal muestran qué clases confundió el modelo. Por ejemplo, una celda de la fila `Class_2` y la columna `Class_4` cuenta productos que eran realmente `Class_4`, pero se predijeron como `Class_2`.

No te quedes solo con el acierto global. Compáralo con la regla mayoritaria y mira también la sensibilidad (*recall*) de `Class_1`: de los productos que realmente son `Class_1`, ¿qué proporción recuperó el modelo? Como este Naive Bayes asigna una clase a todos los productos, su cobertura es 100 %.

Las filas reservadas se consultan en otras comparaciones del cuaderno. Si usamos sus resultados para comparar o escoger variantes, actúan como **validación**, no como un test final independiente.

In [ ]:
# Predice las clases reservadas con el NB ajustado en train.
preds_nb = nb.predict(X_test_nb)  # No consulta las etiquetas reales para decidir.
t_nb = tabla_confusion(y_test_nb, preds_nb, nb.clases)  # Cuenta confusiones entre las nueve clases.
p_acierto = np.mean(preds_nb.to_numpy() == y_test_nb.to_numpy())  # Calcula fracción global de aciertos.
t_nb  # Muestra predicción en filas y clase real en columnas.
dibujar_confusion(t_nb, "Naive Bayes gaussiano · productos reservados")  # Añade la matriz coloreada.
metricas_nb = evaluar(y_test_nb, preds_nb)  # Calcula acierto y cobertura.
recall_minoritaria_nb = t_nb.loc[clase_minoritaria_nb, clase_minoritaria_nb] / t_nb[clase_minoritaria_nb].sum()  # Aciertos / casos reales minoritarios.
display(pd.DataFrame([{  # Resume métricas con denominadores claros.
    "Acierto global (%)": round(100 * p_acierto, 2),  # Porcentaje correcto total.
    "Sensibilidad de clase minoritaria (%)": round(100 * recall_minoritaria_nb, 2),  # Porcentaje de minoritarios detectados.
    "Cobertura (%)": round(100 * metricas_nb["Cobertura"], 2),  # Porcentaje de productos con decisión.
}]))  # Cierra la tabla de evaluación.

#### Leer las probabilidades posteriores

Además de la clase predicha, Naive Bayes puede mostrar un posterior para cada clase. En `probs_nb`, **cada fila es un producto reservado** y **cada columna es una de las nueve clases**. Por ejemplo, un valor 0,70 en la columna `Class_3` significa que, según este modelo y sus supuestos, el posterior asignado a `Class_3` para ese producto es 0,70.

Para clasificar, el modelo elige la clase con posterior más alto. La suma de los nueve posteriores de una fila es aproximadamente 1 porque las puntuaciones se normalizan entre las clases; no significa que el modelo tenga razón con esa frecuencia.

**Posterior no equivale automáticamente a certeza:** la suma 1 es una propiedad del cálculo. Para saber si los valores están bien calibrados habría que comparar probabilidades y frecuencias observadas con un procedimiento específico. Además, la independencia supuesta puede hacer que se cuente evidencia relacionada varias veces. Por eso un posterior alto puede acompañar una predicción incorrecta.

La tabla y el mapa de colores muestran probabilidades ya calculadas. Cambiar el tramo visualizado no vuelve a entrenar el modelo. Compara los posteriores con la clase predicha y con `target` real.

In [ ]:
probs_nb = nb.predict_proba(X_test_nb)  # Calcula el posterior de cada clase para cada producto.
print("Posteriores para los primeros cinco productos reservados:")  # Indica el tramo que se muestra.
display(probs_nb.head().round(3))  # Redondea probabilidades para facilitar lectura.
sumas_posteriores = probs_nb.sum(axis=1).head().rename("Suma de posteriores (≈1)").to_frame()  # Comprueba normalización por fila.
display(sumas_posteriores.round(6))  # Cada producto debe sumar aproximadamente uno.
ejemplos_posterior = pd.DataFrame({  # Reúne clases reales, predichas y posterior máximo.
    "Clase real": y_test_nb.head(),  # Etiqueta verdadera.
    "Clase predicha": preds_nb.head(),  # Etiqueta elegida por el modelo.
    "Posterior máximo": probs_nb.max(axis=1).head(),  # Mayor probabilidad asignada.
}).round(3)  # Redondea las tres columnas.
display(ejemplos_posterior)  # Compara decisiones y respuestas reales.

In [ ]:
from s6_graficos import explorar_probabilidades as _explorar_probabilidades  # Importa el mapa de posteriores por tramo.
from s6_controles import crear_control_probabilidades  # Importa controles de inicio y cantidad.

explorar_probabilidades = partial(_explorar_probabilidades, probabilidades=probs_nb, mapa_colores="viridis")  # Fija la tabla calculada.
control_probabilidades = crear_control_probabilidades(explorar_probabilidades, n_productos=len(probs_nb))  # Ajusta rango de filas.
display(control_probabilidades)  # Muestra el visor interactivo.

#### Sensibilidad por clase

La sensibilidad responde a esta pregunta: **de todos los productos que realmente pertenecen a una clase, qué proporción reconoció el modelo?** Para cada clase se divide el acierto de su diagonal entre el total de su columna, porque las columnas representan las clases reales.

En la gráfica, cada barra es la sensibilidad de una clase y la etiqueta `n=...` indica cuántos productos reales de esa clase hay en las filas reservadas. Ese tamaño importa al interpretar la tasa: una sensibilidad calculada con pocos casos puede variar mucho si cambian unas pocas predicciones.

No confundas sensibilidad con precisión. La sensibilidad parte de los casos **realmente** de esa clase; la precisión partiría de los casos que el modelo **predijo** como esa clase (la fila de la matriz). Compara las nueve barras para ver si el acierto global está ocultando clases que el modelo apenas recupera.

In [ ]:
# Divide los aciertos de cada clase por el total de productos REALES de esa clase.
acierto_por_clase = pd.Series(np.diag(t_nb.to_numpy()) / t_nb.sum(axis=0).to_numpy(),  # Diagonal / total de columna.
                             index=nb.clases, name="Sensibilidad")  # Asocia tasa y clase.
soporte_por_clase = t_nb.sum(axis=0).reindex(nb.clases)  # Cuenta productos reales por categoría.
fig, ax = plt.subplots(figsize=(10, 4.5))  # Crea figura con espacio para nueve barras.
posiciones = np.arange(len(acierto_por_clase))  # Asigna una posición a cada clase.
barras = ax.bar(posiciones, acierto_por_clase.to_numpy(), color="steelblue", alpha=0.85)  # Dibuja sensibilidad.
for barra, sensibilidad_clase, soporte in zip(  # Recorre barra, tasa y tamaño real.
        barras, acierto_por_clase.to_numpy(), soporte_por_clase.to_numpy()):  # Empareja valores por clase.
    _ = ax.annotate(f"{sensibilidad_clase:.0%}\nn={int(soporte):,}".replace(",", " "),  # Escribe tasa y soporte.
                    (barra.get_x() + barra.get_width()/2, barra.get_height()),  # Sitúa texto sobre la barra.
                    xytext=(0, 4), textcoords="offset points", ha="center", va="bottom", fontsize=8)  # Ajusta etiqueta.
_ = ax.set(xticks=posiciones, xticklabels=acierto_por_clase.index, ylim=(0, 1.15),  # Define categorías y escala.
           ylabel="Sensibilidad", title="Acierto por clase real · NB gaussiano")  # Nombra medida y gráfico.
_ = ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda valor, _: f"{valor:.0%}"))  # Muestra eje vertical en porcentaje.
_ = plt.setp(ax.get_xticklabels(), rotation=25, ha="right")  # Gira etiquetas para evitar solapamiento.
_ = ax.grid(axis="y", alpha=0.2)  # Añade líneas guía.
ax.set_axisbelow(True)  # Coloca cuadrícula detrás de las barras.
fig.tight_layout()  # Ajusta márgenes.
plt.show()  # Muestra el gráfico.
display(acierto_por_clase.rename("Sensibilidad (proporción)"))  # Deja disponibles tasas exactas.

**Lectura del Naive Bayes gaussiano base:** los priors se calculan con entrenamiento. Por ejemplo, `Class_2` representa aproximadamente **25,97 %** de los productos de entrenamiento y `Class_1`, **3,18 %**. Son las frecuencias antes de observar las características de un producto; no son los posteriores que aparecen en `probs_nb`.

En las **13 004 filas reservadas**, el modelo acierta **7 494 (57,63 %)**. En cambio, recupera **123 de los 394 productos que realmente son `Class_1`**, es decir, tiene una sensibilidad de **31,22 %** para esa clase. La sensibilidad es **92,03 %** en `Class_5` y **24,88 %** en `Class_3`. Por tanto, el acierto global no describe por igual el rendimiento en todas las clases.

Los ejemplos también muestran que un posterior alto no garantiza una predicción correcta: el producto de índice **7**, cuya clase real es `Class_4`, recibe un posterior máximo de **0,97624** para `Class_2`. El producto de índice **13**, realmente `Class_2`, se asigna a `Class_3` con un posterior cercano a 1. Son resultados del modelo bajo sus supuestos, no certezas.

In [ ]:
try:  # Captura huecos pendientes para que Ejecutar todo pueda continuar.
    posicion_producto_mp = ...  # Sustituye por 1 para consultar la segunda fila (iloc empieza en cero).
    posterior_producto_mp = probs_nb.iloc[posicion_producto_mp]  # Recupera posteriores del producto elegido.
    clase_producto_mp = posterior_producto_mp.idxmax()  # Obtiene la clase con posterior más alto.
    display(pd.DataFrame([{  # Prepara respuesta, predicción, posterior y prior.
        "Clase real": y_test_nb.iloc[posicion_producto_mp],  # Etiqueta verdadera del producto.
        "Clase predicha": clase_producto_mp,  # Etiqueta con posterior máximo.
        "Posterior elegido": posterior_producto_mp.max(),  # Mayor probabilidad posterior.
        "Prior de esa clase": nb.prior.loc[clase_producto_mp],  # Probabilidad de clase antes de observar producto.
    }]).round(4))  # Redondea la tabla del ejemplo.
    clase_recall_mp = ...  # Escribe la etiqueta de la clase minoritaria, no su posición.
    recall_nb_mp = t_nb.loc[clase_recall_mp, clase_recall_mp] / t_nb[clase_recall_mp].sum()  # Aciertos / productos reales.
    display(pd.DataFrame([{"Sensibilidad de la clase elegida (%)": round(100 * recall_nb_mp, 2)}]))  # Muestra recall.
except Exception as error:  # Entra aquí si quedan huecos o nombres incompletos.
    print("Micropráctica pendiente: completa posición, clase y denominador del recall.")  # Indica qué falta.
    print(f"Detalle: {type(error).__name__}: {error}")  # Muestra el error concreto de la celda.

### 6.2.5. Comparar las medias por clase: índice de Gini

Antes de retirar características, usamos un resumen fácil de comparar: para cada variable miramos cuánto difieren sus **medias entre las nueve clases**. El índice de Gini de este cuaderno transforma esa desigualdad en una puntuación:

$$
G_j=\frac{\sum_c\sum_h|\mu_{jc}-\mu_{jh}|}{2C\sum_c\mu_{jc}},\qquad C=9.
$$

Aquí $\mu_{jc}$ es la media de la característica $j$ en la clase $c$, y $C$ es el número de clases. En términos sencillos: si las medias de una característica son muy parecidas entre clases, su Gini será bajo; si unas clases tienen medias bastante mayores que otras, será más alto. Esta puntuación resume **diferencias de medias**, no el acierto del clasificador.

El nombre puede confundir: este índice no es el Gini de un árbol de decisión. Requiere medias no negativas y, si todas son cero, esta implementación devuelve 0. Un valor alto no garantiza que la variable clasifique bien: las distribuciones pueden solaparse mucho, tener gran dispersión o ser redundantes con otras entradas. Tampoco indica causalidad.

Los paneles siguientes muestran las nueve características con mayor y menor puntuación. Cada barra es una media de entrenamiento por clase y el segmento alrededor de ella representa una desviación típica; **no** es un intervalo de confianza. Usa los paneles para describir qué resume el índice, no para concluir por sí solos qué variables conviene usar.

In [ ]:
from s6_seleccion import gini_medias  # Importa el índice que compara medias entre clases.
from s6_graficos import panel_medias as _panel_medias  # Importa los paneles de comparación de medias.

In [ ]:
# Calcula las 93 desigualdades de medias estimadas exclusivamente con entrenamiento.
importancia = nb.medias.apply(gini_medias, axis=0).rename("Gini")  # Aplica Gini a cada columna de medias.
panel_medias = partial(_panel_medias, modelo=nb, importancia=importancia, color="steelblue")  # Fija datos para los gráficos.
print(f"Gini calculado para {len(importancia)} características; criterio descriptivo, no rendimiento del clasificador.")  # Explica el alcance.
print("Diez valores más altos:")  # Identifica el orden de la tabla.
display(importancia.nlargest(10).round(3).to_frame())  # Muestra solo las diez mayores.
print("Diez valores más bajos:")  # Introduce el otro extremo.
display(importancia.nsmallest(10).round(3).to_frame())  # Muestra solo las diez menores.

#### Nueve características con mayor Gini

Estas son las nueve características cuyas **medias de entrenamiento varían más entre clases**, según este índice. Mira el panel para identificar en qué clases son mayores o menores.

Esto no significa que sean automáticamente las nueve mejores predictoras. El Gini solo compara medias: no considera cuánto se solapan los valores de las clases, la dispersión dentro de cada clase, el desequilibrio de frecuencias ni si varias características aportan información repetida. La lista sirve para explorar y para el filtro didáctico del apartado siguiente; su utilidad se juzga comprobando las predicciones.

In [ ]:
mas_predictoras = importancia.nlargest(9).index.tolist()  # Guarda las nueve características con mayor Gini.
tabla_mayor_gini = importancia.loc[mas_predictoras].sort_values(ascending=False).round(3).to_frame()  # Ordena y prepara tabla.
display(tabla_mayor_gini)  # Muestra nombres y valores del índice.
panel_medias(mas_predictoras, "Mayor desigualdad entre medias de clase")  # Compara sus medias y desviaciones por clase.

### 6.2.6. Seleccionar entradas y comparar modelos

Ahora hacemos dos experimentos sencillos: entrenamos Naive Bayes con todas las características y lo comparamos con versiones que conservan menos entradas. La pregunta es si una regla simple de selección mantiene o cambia el comportamiento del modelo; no damos por hecho que reducir entradas lo mejore.

1. **Gini > 0,5:** conserva las características cuyas medias difieren suficientemente entre las clases según el índice anterior. El umbral 0,5 es una elección fija del ejercicio, no un valor universal.
2. **Correlación positiva > 0,8:** si dos características tienen una asociación lineal positiva muy alta, el filtro retira todas las que participan en al menos una pareja así. Puede retirar ambas, no elige una representante. Se usa $r>0,8$, no $|r|>0,8$; por tanto, una correlación negativa fuerte no activa esta regla.

Después comparamos el acierto global y la sensibilidad de `Class_1` en las mismas filas reservadas. Estos filtros no garantizan una mejora: el Gini solo mira medias, y la correlación marginal entre entradas no demuestra dependencia condicional dentro de las clases. Además, la matriz de correlación se calculó con toda la muestra, incluidas las filas reservadas; por ello, el resultado del segundo filtro es **exploratorio**, no una evaluación final independiente.

In [ ]:
# El histograma cuenta características; la línea discontinua muestra el umbral fijo.
fig, ax = plt.subplots(figsize=(8, 4))  # Crea figura con tamaño legible.
_ = ax.hist(importancia, bins=np.linspace(0, max(0.6, importancia.max()), 13),  # Agrupa valores de Gini en intervalos.
            color="steelblue", edgecolor="white")  # Distingue visualmente las barras.
_ = ax.axvline(0.5, color="firebrick", linestyle="--", linewidth=1.5,  # Dibuja el corte del ejercicio.
               label=f"Umbral 0,5 · {(importancia > 0.5).sum()} variables")  # Cuenta variables por encima.
_ = ax.set(xlabel="Gini de las medias", ylabel="Número de variables",  # Nombra los ejes.
           title="Desigualdad de medias por característica")  # Describe el gráfico.
_ = ax.grid(axis="y", alpha=0.2)  # Añade guías para leer alturas.
ax.set_axisbelow(True)  # Coloca cuadrícula detrás de las barras.
_ = ax.legend(frameon=False)  # Muestra el significado del umbral.
fig.tight_layout()  # Ajusta los márgenes de la figura.
plt.show()  # Muestra el gráfico.

In [ ]:
# Selección original con umbral fijo Gini > 0,5.
seleccion_gini = importancia.index[importancia > 0.5].tolist()  # Guarda variables que superan el corte.
train1 = train_nb[seleccion_gini + ["target"]]  # Mantiene target como respuesta, no predictor.
nb1 = BayesGaussiano().fit(train1.drop(columns="target"), y_train_nb)  # Ajusta NB con variables seleccionadas.
preds_nb1 = nb1.predict(X_test_nb)  # Predice las etiquetas reservadas.
t_nb1 = tabla_confusion(y_test_nb, preds_nb1, nb.clases)  # Cuenta confusiones para todas las clases.
recall_class1_nb1 = t_nb1.loc["Class_1", "Class_1"] / t_nb1["Class_1"].sum()  # Calcula sensibilidad de Class_1.
display(pd.DataFrame([{  # Resume el tamaño del modelo y sus métricas.
    "Variables seleccionadas": len(seleccion_gini),  # Cantidad de predictores conservados.
    "Acierto global (%)": round(100 * preds_nb1.eq(y_test_nb).mean(), 2),  # Porcentaje de productos correctos.
    "Sensibilidad Class_1 (%)": round(100 * recall_class1_nb1, 2),  # Porcentaje de Class_1 recuperada.
}]))  # Cierra la tabla resumen.
display(importancia.loc[seleccion_gini].nlargest(10).round(3).to_frame())  # Enseña las diez mayores puntuaciones Gini.

#### Filtro de correlación: retirar los extremos

La matriz de correlación resume asociaciones lineales entre pares de características. Aquí fijamos una regla concreta: marcar cada pareja con **correlación positiva mayor que 0,8** y retirar toda característica que aparezca en al menos una pareja marcada.

Por ejemplo, si `feat_a` y `feat_b` superan el umbral, se retiran ambas; este filtro **no** decide cuál conservar. Tampoco usa el valor absoluto: dos características con correlación negativa muy fuerte no se retiran con esta regla.

La matriz se calculó sobre toda la muestra, no solo entrenamiento. Por eso las filas reservadas influyeron en qué columnas se retiraron y el resultado debe leerse como una demostración exploratoria, no como una estimación imparcial de cómo rendiría el filtro en datos nuevos. Retirar correlaciones tampoco demuestra que se cumpla la independencia condicional de Naive Bayes.

In [ ]:
# Conserva la regla positiva r > 0,8; la diagonal no cuenta como otra característica.
cor_altas = cor_productos.gt(0.8).copy()  # Marca parejas que superan el umbral.
cor_altas = cor_altas.mask(np.eye(len(cor_altas), dtype=bool), False)  # Quita autocorrelaciones de la diagonal.
numero_cor_altas = cor_altas.sum(axis=0).sort_values(ascending=False)  # Cuenta parejas altas por característica.
retiradas_cor = numero_cor_altas.index[numero_cor_altas >= 1].tolist()  # Selecciona características implicadas.

# Muestra solo las características afectadas y resume el tamaño del filtro.
tabla_cor_altas = numero_cor_altas.loc[numero_cor_altas > 0].rename(  # Oculta variables no implicadas.
    "Parejas con r > 0,8"  # Da nombre al recuento de relaciones altas.
).to_frame()  # Convierte el resultado en tabla.
tabla_cor_altas.index.name = "Característica"  # Nombra el índice de filas.
display(tabla_cor_altas)  # Muestra solo las variables retiradas.
print(f"Se retiran {len(retiradas_cor)} de {len(predictores_nb)} características.")  # Resume el tamaño del filtro.
train2 = train_nb.drop(columns=retiradas_cor)  # Prepara train con entradas restantes; target se conserva.

In [ ]:
nb2 = BayesGaussiano().fit(train2.drop(columns="target"), y_train_nb)  # Ajusta NB sin entradas retiradas.
preds_nb2 = nb2.predict(X_test_nb)  # Predice los productos reservados.
t_nb2 = tabla_confusion(y_test_nb, preds_nb2, nb.clases)  # Cuenta errores y aciertos por clase.
recall_class1_nb2 = t_nb2.loc["Class_1", "Class_1"] / t_nb2["Class_1"].sum()  # Calcula recall de Class_1.
display(pd.DataFrame([{  # Resume tamaño del modelo y rendimiento.
    "Variables retiradas": len(retiradas_cor),  # Número eliminado por el filtro.
    "Variables conservadas": len(nb2.variables),  # Número que ajustó el modelo.
    "Acierto global (%)": round(100 * preds_nb2.eq(y_test_nb).mean(), 2),  # Porcentaje correcto.
    "Sensibilidad Class_1 (%)": round(100 * recall_class1_nb2, 2),  # Porcentaje de Class_1 detectada.
}]))  # Termina la tabla resumen.
display(pd.DataFrame({"Característica retirada": retiradas_cor}))  # Enumera las variables eliminadas.

### Comparar las variantes NB

El cuadro compara el Naive Bayes gaussiano base con los dos modelos reducidos de los apartados anteriores. Las tres predicciones se calcularon para **los mismos productos reservados**, con la misma respuesta real; así podemos comparar sus cifras directamente. Esta celda solo reúne resultados, no vuelve a entrenar los modelos.

- **Accuracy (acierto global):** número de productos clasificados correctamente dividido por los 13 004 productos evaluados.
- **Recall de `Class_1` (sensibilidad):** productos `Class_1` correctamente recuperados dividido por los **394 productos que realmente son `Class_1`**. El denominador es el mismo para los tres modelos.

Mira ambas columnas: un modelo puede mantener un acierto global parecido y recuperar una proporción distinta de la clase minoritaria. Las filas reservadas se han consultado durante varias comparaciones, por lo que estos resultados son de **evaluación/validación**, no una prueba final sobre datos nuevos. No elijas un ganador solo con esta tabla.

In [ ]:
from s6_evaluacion import filas_comparacion_variantes  # Resume modelos ya ajustados, sin volver a entrenarlos.
variantes_nb = [  # Define las tres variantes de Naive Bayes que se comparan.
    ("Base gaussiano", nb, preds_nb),  # Referencia con todas las entradas y gaussianas.
    ("Gini > 0,5", nb1, preds_nb1),  # Modelo con filtro por separación de medias.
    ("Correlación > 0,8: retirar extremos", nb2, preds_nb2),  # Modelo sin extremos correlacionados.
]  # Cierra la lista de modelos y etiquetas calculados.
reales_class1_nb = y_test_nb.eq("Class_1")  # Identifica productos reales de Class_1.
assert int(reales_class1_nb.sum()) == 394  # Confirma denominador usado para recall.
filas_comparacion_nb = filas_comparacion_variantes(variantes_nb, y_test_nb, clase_minoritaria="Class_1")  # Calcula resumen.
comparacion_nb = pd.DataFrame(filas_comparacion_nb).set_index("Variante")  # Organiza resultados por nombre de modelo.
display(comparacion_nb.round({"Accuracy (%)": 2, "Recall Class_1 (%)": 2}))  # Muestra acierto y recall redondeados.

# El gráfico contrasta acierto global y detección de la clase minoritaria.
fig, ax = plt.subplots(figsize=(8, 4))  # Prepara un gráfico compacto para tres variantes.
comparacion_grafico = comparacion_nb[["Accuracy (%)", "Recall Class_1 (%)"]]  # Selecciona dos métricas.
posiciones = np.arange(len(comparacion_grafico))  # Asigna posición horizontal a cada variante.
ancho = 0.38  # Define ancho para agrupar dos barras por método.
_ = ax.bar(posiciones - ancho/2, comparacion_grafico.iloc[:, 0], ancho,  # Dibuja acierto global a la izquierda.
           label="Acierto global", color="steelblue")  # Identifica color y métrica.
_ = ax.bar(posiciones + ancho/2, comparacion_grafico.iloc[:, 1], ancho,  # Dibuja recall minoritario a la derecha.
           label="Sensibilidad Class_1", color="darkorange")  # Identifica color y métrica.
_ = ax.set(xticks=posiciones, xticklabels=comparacion_grafico.index, ylim=(0, 100),  # Configura etiquetas y rango.
           ylabel="Porcentaje", title="Acierto global y recuperación de Class_1")  # Explica los ejes.
_ = ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda valor, _: f"{valor:.0f}%"))  # Añade símbolo de porcentaje.
_ = plt.setp(ax.get_xticklabels(), rotation=18, ha="right")  # Gira etiquetas para facilitar lectura.
_ = ax.grid(axis="y", alpha=0.2)  # Añade guías horizontales.
ax.set_axisbelow(True)  # Coloca cuadrícula detrás de las barras.
_ = ax.legend(frameon=False)  # Muestra leyenda.
fig.tight_layout()  # Evita recortes de etiquetas.
plt.show()  # Dibuja el gráfico.
nombre_variante_nb, modelo_variante_nb, etiquetas_variante_nb = variantes_nb[-1]  # Guarda última variante para inspección.
aciertos_variante_nb = etiquetas_variante_nb.eq(y_test_nb)  # Marca aciertos de la última variante.

**Lectura de la comparación:** el NB gaussiano base obtiene **57,63 %** de acierto y recupera **123/394** productos de `Class_1` (**31,22 %**). La regla mayoritaria acierta **26,05 %**.

- **Gini > 0,5:** conserva **49 entradas**; obtiene **45,29 %** de acierto y **18,78 %** de sensibilidad para `Class_1` (**74/394**).
- **Correlación > 0,8:** conserva **91 entradas**; obtiene **57,54 %** de acierto y **31,73 %** de sensibilidad (**125/394**). El resultado apenas cambia frente al NB base.

No hay una mejora única ni una explicación causal de las diferencias. El filtro de correlación se calculó sobre toda la muestra; por eso esta comparación es exploratoria, no una prueba final sobre datos independientes.

## Ejercicio 6-2 — KNN y combinación con NB (sin resolver)

**Objetivo:** comprobar si KNN añade algo al Naive Bayes de productos y explorar una combinación sencilla de ambos. Trabajarás con la misma partición `train_nb` / `test_nb` para que las comparaciones usen las mismas filas. No uses `id` como predictor ni `target` como entrada.

### 1. Seleccionar 20 entradas y estudiar KNN

Elige con datos de entrenamiento un criterio para quedarte con **20 características**. Explica qué mide ese criterio y qué información puede dejar fuera. Aprende las escalas también solo con `X_train_nb` y aplica los mismos parámetros a entrenamiento y evaluación; así evitas que las filas reservadas influyan en la transformación.

Ajusta una búsqueda `Vecinos` con las 20 entradas y compara varios valores de `k`. Informa del acierto, la matriz de confusión y la sensibilidad de `Class_1`; usa la regla mayoritaria como referencia. Si pruebas varios valores de `k` y eliges uno mirando las etiquetas reservadas, estas filas son validación, no test final.

**Consulta:** selección de entradas en **6.2.5–6.2.6** y escalado/voto KNN en **6.1.9**. Las secciones anteriores son ejemplos: la selección concreta y la solución del ejercicio quedan a tu cargo.

In [ ]:
# Apartado 1 — Seleccionar 20 entradas y estudiar KNN.
# Consulta selección en 6.2.5–6.2.6 y escalas/votos en 6.1.9.
# Elige el criterio con entrenamiento; conserva las mismas filas e índices.
# Aprende las escalas solo con X_train_nb y aplícalas a ambas tablas.
# Crea un Vecinos independiente y reutiliza barrido_vecinos para varios k.
# Compara acierto, matriz, referencia mayoritaria y sensibilidad de Class_1.
# Escribe aquí tu solución.

### 2. Promediar probabilidades de KNN y Naive Bayes

Como segundo experimento, combina las salidas probabilísticas de los dos métodos. Obtén para **exactamente las mismas filas y en el mismo orden** los votos por clase de KNN y los posteriores de Naive Bayes (`probs_nb`). Calcula su promedio simple, sin ajustar pesos.

Antes de promediar, verifica que ambas tablas tienen los mismos índices y las mismas columnas de clase. Después comprueba que las filas de cada tabla suman aproximadamente 1; repite la comprobación para el promedio. La clase combinada será la que tenga el promedio más alto.

Por último, compara el acierto y la sensibilidad de `Class_1` con KNN y Naive Bayes por separado. Pregúntate si la combinación ayuda a la clase minoritaria o solo cambia el acierto global. Un promedio que suma 1 no garantiza que esté calibrado ni que mejore el resultado.

**Importante:** si consultas las etiquetas reservadas para decidir si la combinación es mejor o para ajustar pesos, esas filas se convierten en validación. No las presentes como una evaluación final independiente.

In [ ]:
# Apartado 2 — Promediar las probabilidades y evaluar la asignación.
# Obtén los votos de KNN para exactamente X_test_nb, con el mismo orden de productos.
# Alinea índices y columnas con probs_nb; comprueba la normalización.
# Asigna la clase de mayor probabilidad media.
# Compara acierto y sensibilidad de Class_1 con ambos modelos por separado.
# No optimices pesos usando las respuestas reservadas.
# Escribe aquí tu solución.

## Cierre — sesión 6

- **KNN:** clasifica mediante distancias y votos. Cambiar `k`, los pesos o la escala cambia la regla y la geometría; no garantiza una mejora.
- **Transformaciones:** centros, desviaciones y cuantiles se aprenden con entrenamiento. La cuantificación por rangos no convierte los códigos nominales en distancias naturales.
- **Naive Bayes:** combina priors con distribuciones por clase y mantiene el supuesto de independencia condicional. Normalizar posteriores no demuestra certeza ni calibración.
- **Evaluación:** el acierto global puede ocultar errores de la clase minoritaria. El NB base obtiene **57,63 %** de acierto y recupera **123/394** productos de `Class_1` (**31,22 %**).
- **Validación y test:** `test_knn` y `test_nb` se consultan en varias comparaciones, así que actúan como evaluación/validación. Un test final independiente solo se consulta cuando las decisiones están fijadas; los filtros también deben aprenderse sin sus filas.

**Los ejercicios 6-1 y 6-2 permanecen sin resolver.** No se ejecutan sus funciones pendientes ni se declara un ganador a partir de las comparaciones de este cuaderno.